<!-- notebook-header -->
<a id="top"></a>

# 7. Heaps, Intervals & Greedy

*LeetCode Playbook · notebook 7 of 12* · Always the cheapest next, overlapping ranges, and local choices you can prove are safe.

**In this notebook:** [Heaps](#s13) · [Intervals & Sweep Line](#s14) · [Greedy](#s15)

← [Trees & Tries](06_Trees_Tries.ipynb) · [All notebooks](01_Start_Here.ipynb#notebooks) · [A-Z problem finder](12_Interview_Day.ipynb#finder) · [Backtracking (and a short DP map)](08_Backtracking.ipynb) →

Run the setup cell below first; after that every section runs from its first cell.

In [ ]:
# Setup: everything this notebook imports. Run this cell first.
from collections import Counter, defaultdict, deque, OrderedDict
from functools import cmp_to_key, lru_cache
from itertools import accumulate, combinations, permutations, product
import bisect, heapq, math, random, string

<a id="s13"></a>

## Heaps

> A heap is a pile that always hands you the smallest item first. It keeps only a *partial* order (every parent ≤ its children), which is exactly enough to read the minimum in O(1) and to add or remove an item in O(log n).

**Reach for it when** you need the smallest or largest item *again and again* while items keep arriving or leaving: **top k, k-th largest, k closest**, **merge k sorted** lists, the **median of a stream**, "always process the cheapest / earliest / most frequent next", scheduling with deadlines or cooldowns, or growing a frontier from its lowest point (Dijkstra-style).

**In this repo:** `heap/` (20 problems) · bank: `practice/simple/32_k_closest_points_to_origin.py`, `practice/simple/33_task_scheduler.py`, `practice/simple/34_find_median_from_data_stream.py`, `practice/simple/35_merge_k_sorted_lists.py` · basics: `practice/simple/basics/heaps/` (`01_heapify_by_hand.py`, `02_push_and_pop_by_hand.py`, `03_top_k_with_size_k_min_heap.py`, `04_max_heap_by_negation_and_tuples.py`, `05_kth_largest_in_a_stream.py`, `06_merge_k_sorted_arrays.py`), `practice/simple/basics/sorting/04_heap_sort.py`

### The picture

```text
the list:  [1, 3, 2, 7, 4, 9, 5]       node i has children 2i+1 and 2i+2, parent (i-1)//2

the tree:          1                   the only rule: every parent <= its children
                 /   \                 -> the smallest is always heap[0]
                3     2                -> nothing else is in order: heap[1] = 3 is not the
               / \   / \                  second smallest (2 is), heap[-1] = 5 is not the
              7   4 9   5                 largest (9 is)

push 0:  append it as the last leaf, then swap it UP while its parent is bigger
         [1, 3, 2, 7, 4, 9, 5, 0]  ->  0 passes 7, 3, 1  ->  [0, 1, 2, 3, 4, 9, 5, 7]
pop:     take heap[0], move the last leaf to the root, swap it DOWN toward its smaller child
         both walk a single root-to-leaf path: at most log2(n) swaps
```

**Why it is fast:** the brute-force answer to "give me the smallest" is a scan, O(n) every time you ask, or a full sort, O(n log n), most of whose order you never use. A heap keeps just enough order to answer one question (what is the smallest?) and repairs it along one path after each change: O(1) to peek, O(log n) to push or pop. Capped at k items, it answers "top k of n" in O(n log k) time and O(k) memory, even on a stream you can't store. `heapify` builds a heap from a whole list in O(n), not O(n log n): it sifts each parent *down*, and a node can only sink as far as its height. Half the nodes are leaves that don't move, a quarter sink at most one level, an eighth at most two, and that sum stays below n.

### heapq in one cell

Python's `heapq` works on a plain list, and its classic functions give you a **min**-heap. Everything else (max-heaps, priorities, ties) is done with what you push.

In [ ]:
h = []
for x in [5, 1, 2, 8]:
    heapq.heappush(h, x)                     # O(log n): append, then sift up
print(h, h[0])                               # [1, 5, 2, 8] 1   (only h[0] has a meaning)
print(heapq.heappop(h), h)                   # 1 [2, 5, 8]

nums = [4, 1, 7, 3]
heapq.heapify(nums)                          # O(n), in place, returns None
print(nums)                                  # [1, 3, 7, 4]

mx = [-x for x in [4, 1, 7, 3]]              # max-heap: store -x ...
heapq.heapify(mx)
print(-heapq.heappop(mx), -mx[0])            # 7 4   ... and negate on the way out

jobs = [(2, "write"), (1, "deploy"), (2, "test")]
heap = []
for order, (prio, name) in enumerate(jobs):
    heapq.heappush(heap, (prio, order, name))   # tuples compare left to right: order breaks ties
print([heapq.heappop(heap)[2] for _ in range(3)])   # ['deploy', 'write', 'test']

try:
    heapq.heappush([(1, {"id": 1})], (1, {"id": 2}))   # equal priorities -> Python compares the dicts
except TypeError as e:
    print("TypeError:", e)                   # '<' not supported between instances of 'dict' and 'dict'
print(heapq.nlargest(2, [4, 1, 7, 3]), heapq.nsmallest(2, [4, 1, 7, 3]))   # [7, 4] [1, 3]
print(list(heapq.merge([1, 4, 5], [1, 3, 4], [2, 6])))    # [1, 1, 2, 3, 4, 4, 5, 6]  (lazy k-way merge)

**Try it**
- Print `h[1]` and `sorted(h)[1]` right after the four pushes: 5 versus 2. Only `h[0]` is guaranteed; to read the k smallest in order, pop k times or sort.
- Push `(prio, name)` without `order`: it still runs (names are strings), but the tie now comes out alphabetically, `['deploy', 'test', 'write']`, not in arrival order.
- Push `(1, {"id": 1})` and then `(2, {"id": 2})` into an empty heap: no error. The dicts are only compared when two priorities tie, which is why this bug hides until the first tie.
- Predict, then run: `heapq.heappushpop([2, 5, 8], 0)` returns 0 (push first, so 0 comes straight back out) and `heapq.heapreplace([2, 5, 8], 0)` returns 2 (pop first, then push).

### From idea to code

**The idea in one sentence:** *keep the candidates in a heap keyed by "who should go next"; pop the best, deal with it, push any new candidates it creates; and when only the best k matter, let the heap throw out its weakest member whenever it holds k + 1.*

**Why a min-heap for the k *largest*?** Each newcomer asks one question: "am I better than the weakest of the k?" The weakest of the k largest is their *minimum*, so the minimum must sit on top. A max-heap of all n items would also work (pop k times), but it holds all n: O(n) memory, and useless for a stream.

| Decision | Keep the k best (215, 703, 973) | Next from a frontier (23, 632, 407) |
|---|---|---|
| **State / Definition** | `heap` = min-heap of the k largest so far; `heap[0]` = the weakest of them | `heap` = one candidate per source that could come next, e.g. `(value, list i, index j)` |
| **Invariant** | after the fix, the heap holds the k largest items seen so far (all of them while fewer than k have arrived) | every item not yet popped is in the heap or waits behind one (later in the same list), so the root is the next item overall |
| **Step** | push the newcomer | pop the root: it is the next item overall |
| **Fix** | `if len(heap) > k: heappop(heap)`: the weakest of k + 1 leaves, maybe the newcomer itself | push the popped item's successor: its list's next item, its unvisited neighbours |
| **Record** | `heap[0]` is the k-th largest (in Kth Largest in a Stream: `return heap[0]` at the end of `add`) | right after each pop, because pops come out in order (or when an item is first reached, if its value is final then: Trapping Rain Water II banks water at the push) |
| **Init** | `heap = []` | `heapify` one head per list, or the whole border, in O(n) |
| **Return** | the heap's items (in heap order: sort them if order matters) or `heap[0]` | the pops, or the value recorded when the target shows up; needing a pop from an empty heap means "impossible" (−1, `""`) |

The same idea, sentence by sentence:

| In words | In code |
|---|---|
| "the smallest candidate" (just look) | `heap[0]` |
| "take the smallest out" | `heapq.heappop(heap)` |
| "add a candidate; ties go to the earlier one" | `heapq.heappush(heap, (key, i, item))` |
| "the largest instead" | push `-key`; the largest is `-heap[0]` |
| "keep only the k largest" | `heappush(heap, x)`, then `if len(heap) > k: heappop(heap)` |
| "the k-th largest so far" | `heap[0]` of that size-k min-heap |
| "push, then pop" / "pop, then push" (one call) | `heapq.heappushpop(heap, x)` / `heapq.heapreplace(heap, x)` |
| "the next item from the list I just used" | `if j + 1 < len(lists[i]): heappush(heap, (lists[i][j + 1], i, j + 1))` |

**Push first, then evict.** The heap compares the newcomer with the weakest of the k for you: if the newcomer is the weakest of the k + 1, it is the one that leaves. Evicting first throws out a kept item before anyone has checked that the newcomer is better. In the merge the order is pop, record, refill: only after the pop do you know which list needs a new head.

In [ ]:
def k_largest(nums, k):
    heap = []                                # STATE + INIT: min-heap of the k largest so far; heap[0] = the weakest
    for x in nums:
        heapq.heappush(heap, x)              # STEP: x joins the candidates
        if len(heap) > k:                    # FIX: k + 1 candidates, the weakest leaves (maybe x itself)
            heapq.heappop(heap)
    return sorted(heap, reverse=True)        # RETURN: the k largest, largest first


def merge_sorted(lists):
    heap = [(lst[0], i, 0) for i, lst in enumerate(lists) if lst]   # STATE + INIT: (value, list i, index j), one head per list
    heapq.heapify(heap)
    out = []
    while heap:
        val, i, j = heapq.heappop(heap)      # STEP: the smallest head is the next item overall
        out.append(val)                      # RECORD: pops come out in sorted order
        if j + 1 < len(lists[i]):            # FIX: list i lost its head; its next item takes the seat
            heapq.heappush(heap, (lists[i][j + 1], i, j + 1))
    return out                               # RETURN


print(k_largest([3, 1, 5, 12, 2, 11], 3))            # [12, 11, 5]
print(merge_sorted([[1, 4, 5], [1, 3, 4], [2, 6]]))  # [1, 1, 2, 3, 4, 4, 5, 6]

**Try it**
- Change `if len(heap) > k:` to `>=` and rerun: `[12, 11]`. The heap is now capped at k − 1.
- Evict first: replace the two lines in the loop with `if len(heap) == k: heapq.heappop(heap)` followed by `heapq.heappush(heap, x)`. `k_largest([5, 1], 1)` returns `[1]`: the 5 was thrown out before anyone checked that 1 is worse.
- Print `heap` at the top of the `while` loop in `merge_sorted`: it never holds more than 3 entries, one per list. That is where O(N log k) comes from.
- Remove `if lst` from the first line of `merge_sorted` and call `merge_sorted([[], [2]])`: `IndexError`, because an empty list has no head.

### Watch it work

The trace runs the template's own two lines, push then evict, and says who left. Once the heap holds k items, its root is a doorman: a newcomer stays only if it beats the root, and then the root leaves.

In [ ]:
def trace_k_largest(nums, k):
    heap = []
    for x in nums:
        heapq.heappush(heap, x)                                   # STEP
        gone = heapq.heappop(heap) if len(heap) > k else None     # FIX
        note = ("filling up" if gone is None else
                "bounced: x was the weakest" if gone == x else f"{gone} leaves")
        print(f"x={x:<3} heap={str(heap):<12} root={heap[0]:<3} {note}")


trace_k_largest([3, 1, 5, 12, 2, 11], 3)

**Try it**
- Run it with `k = 1`: the heap is a single "best so far", and the root only climbs (3, then 5, then 12).
- Run `[1, 2, 3, 4, 5]` and then `[5, 4, 3, 2, 1]` with `k = 2`: once the heap is full, every newcomer stays on the rising list and every newcomer bounces on the falling one. Same O(n log k) bound, very different traffic.
- Rewrite the FIX compare-first: `if len(heap) < k: heapq.heappush(heap, x)` / `elif x > heap[0]: heapq.heapreplace(heap, x)`. The `root` column is the same at every step, but a newcomer that can't beat the root never enters. (With `>=` there, `[5, 5, 5, 5]`, k = 2 swaps equal values in and out for nothing.)
- Read the `heap=` column: it need not be sorted (`[3, 12, 5]`), yet once the heap is full, `root` is always the k-th largest so far.

### Where it goes wrong

1. **Ties compare the payload.** `(dist, node)` raises `TypeError` the first time two distances are equal and the nodes can't be compared: that is the `{"id": ...}` error in the heapq cell. Push `(dist, i, node)` with a unique `i`, the list index or a running counter.
2. **Negating only half of the time.** Push `-x`, read `-heap[0]`, negate what you pop. In the median finder below, dropping the minus in step 2 (`heappush(self.high, heappop(self.low))`) makes `median()` return −5.0 after `add(5)`. A missing minus gives a silently wrong answer, never an error.
3. **The wrong heap for top-k.** k largest → *min*-heap capped at k; k smallest or k closest → *max*-heap capped at k. The other way round evicts your best items: `[3, 1, 5]`, k = 1, with a max-heap keeps 1.
4. **Reading more than the root.** `heap[1]` is not the second smallest (`[1, 5, 2, 8]` above) and `heap[-1]` is not the largest. Pop or sort to read in order.
5. **Changing a key that is already in the heap.** The heap never notices: `h = [[1, "a"], [2, "b"]]; h[1][0] = 0; heapq.heappop(h)` returns `[1, "a"]`. Push a fresh entry instead and skip the stale one when it surfaces (lazy deletion: right after a pop, ask "is this entry still valid?").
6. **Popping an empty heap.** `IndexError`. IPO below crashes on `max_capital(1, 0, [5], [1])` without its `if not heap: break`. An empty heap is often the "impossible" answer.
7. **`heappush` onto a list that is not a heap.** `heapq` never checks: `h = [5, 1]; heapq.heappush(h, 3)` leaves `h[0] == 3`, not 1. Call `heapify` once first.
8. **Using what `heapify` returns.** It works in place and returns `None`: `h = heapq.heapify([3, 1]); h[0]` raises `TypeError: 'NoneType' object is not subscriptable`.
9. **Two successors per pop.** In a grid of sorted sums (373, 378), pushing both `(i + 1, j)` and `(i, j + 1)` after every pop reaches the same cell twice: with `nums1 = nums2 = [1, 2]` and k = 5, the pair `[2, 2]` comes out twice. Seed one head per row and only move right, or keep a `seen` set.

### Edge cases to say out loud

k = 0 · k ≥ n · duplicates (they count separately) · empty lists among the k lists · all lists empty · equal keys with uncomparable payloads · negatives under negation · an even count for the median (average, as a float) · the heap running dry before you are done (impossible input).

In [ ]:
assert k_largest([5, 5, 5], 2) == [5, 5]                  # duplicates count separately
assert k_largest([2, 1], 5) == [2, 1]                     # k >= n: keep everything
assert k_largest([4, 1], 0) == []                         # k = 0: everything leaves again
assert k_largest([], 3) == []
assert k_largest([-1, -7, -3], 2) == [-1, -3]
assert merge_sorted([]) == [] and merge_sorted([[], []]) == []
assert merge_sorted([[1, 1], [1]]) == [1, 1, 1]           # equal heads: the list index breaks the tie
assert merge_sorted([[-3, 0], [-5], [7]]) == [-5, -3, 0, 7]
print("edge cases pass")

**Try it**
- Predict, then add: `assert k_largest([2, 9, 4], 3) == [9, 4, 2]` (k = n gives the input sorted, largest first).
- Why does `merge_sorted([[1, 1], [1]])` never compare anything but numbers? Print the heap after `heapify`: `[(1, 0, 0), (1, 1, 0)]`; the second field already differs.
- LeetCode 23 passes linked-list nodes: push `(node.val, i, node)` and refill with `node.next`. Drop the `i` and merge two lists whose heads are equal: `TypeError`, because Python then compares two `ListNode`s.

### Variations

| Variation | What changes from the template | Problems |
|---|---|---|
| **Keep the k best** | min-heap capped at k for the k largest; max-heap (negate) for the k smallest / closest; 1383 sorts by the bottleneck, then keeps the k best speeds | 215, 703, 973, 1383 |
| **Custom order** | strings can't be negated: a class with `__lt__` ("less" = worse), or heapify all `(-count, word)` and pop k | 692 |
| **Two heaps** | max-heap for the low half, min-heap for the high half; the median sits at the roots | 295, 480 |
| **Cooldown** | max-heap of counts + a FIFO queue (or one held item) of what can't be used yet | 621, 767, 358 |
| **K-way merge** | the template: one head per sorted list; pop the smallest, push its successor | 23, 355, 786 |
| **Two orders at once** | one heap of free rooms by id, one of busy rooms by end time | 2402 |
| **Shrink the max** | max-heap; only lowering the max can shrink max − min; track the min beside it | 1675 |
| *Stretch:* **unlock, then take the best** | sort by the unlock key, push everything unlocked, pop the best when you must choose | 502, 871, 1834 |
| *Stretch:* **take now, regret later** | take every item; when a limit breaks, pop the worst item you took | 630, 1642 |
| *Stretch:* **lazy deletion** | leave dead entries in; pop them only when they reach the top | 218, 480, 1851 |
| *Stretch:* **lowest frontier first** | heap of frontier cells keyed by level or distance (Dijkstra shape) | 407, 743 |
| *Stretch:* **merge + running max** | the range `[heap min, running max]` covers every list; advance the min | 632 |

**Keep k, mirror image.** For the k *closest*, cap a *max*-heap at k: its root is the farthest point you kept, and a newcomer stays only if it is closer. Words can't be negated, so Top K Frequent Words gives its entries a `__lt__` where "less" means "worse"; then the same push-and-evict loop works. (Simpler when k is small: heapify all `(-count, word)` and pop k times, O(m + k log m).)

In [ ]:
def k_closest(points, k):
    heap = []                                # STATE: max-heap via (-dist, x, y); root = the farthest kept
    for x, y in points:
        heapq.heappush(heap, (-(x * x + y * y), x, y))   # STEP: squared distance, same order, no sqrt
        if len(heap) > k:                    # FIX: the farthest of k + 1 leaves
            heapq.heappop(heap)
    return sorted([x, y] for _, x, y in heap)            # RETURN (sorted only for a stable printout)


class Entry:                                 # "less" means "worse": fewer copies, or same count and later word
    def __init__(self, count, word):
        self.count, self.word = count, word

    def __lt__(self, other):
        if self.count != other.count:
            return self.count < other.count
        return self.word > other.word


def top_k_words(words, k):
    heap = []                                # STATE: min-heap of the k best words; root = the worst kept
    for word, count in Counter(words).items():
        heapq.heappush(heap, Entry(count, word))         # STEP
        if len(heap) > k:                    # FIX: the worst of k + 1 leaves
            heapq.heappop(heap)
    return [e.word for e in sorted(heap, reverse=True)]  # RETURN: best first


print(k_closest([[3, 3], [5, -1], [-2, 4]], 2))                            # [[-2, 4], [3, 3]]
print(top_k_words(["i", "love", "leetcode", "i", "love", "coding"], 2))    # ['i', 'love']

**Try it**
- Drop the minus in `k_closest` (push `(x * x + y * y, x, y)`): you get the 2 *farthest* points, `[[-2, 4], [5, -1]]`, because the root is now the closest point and it is the one evicted.
- Run `top_k_words(["b", "a", "b", "a"], 1)`: `['a']`. Both words appear twice, and on a tie the later word, `'b'`, is the "worse" one at the root.
- Flip the tie rule in `__lt__` to `self.word < other.word` and rerun that call: `['b']`, the wrong word. The root must be the word you would throw out first.
- `k_closest([[1, 1]], 5)` returns `[[1, 1]]`: the cap is simply never reached.

**Two heaps (running median).** The median lives at the seam between the smaller half and the larger half. Keep the smaller half in a max-heap and the larger half in a min-heap, and the middle numbers sit at the two roots.

```text
   low (max-heap, stored negated)        high (min-heap)
   1   3   [5]          |                [15]   20
             ^ largest small number       ^ smallest large number
   invariant: every low <= every high, and len(low) is len(high) or len(high) + 1
   median: -low[0] when the count is odd (here 5), else the mean of the two roots
```

In [ ]:
class MedianFinder:
    def __init__(self):
        self.low = []                        # STATE: max-heap (stored negated), the smaller half
        self.high = []                       # STATE: min-heap, the larger half

    def add(self, x):
        heapq.heappush(self.low, -x)                         # STEP: 1. x joins the small half
        heapq.heappush(self.high, -heapq.heappop(self.low))  # FIX: 2. the largest small crosses the seam
        if len(self.high) > len(self.low):                   # FIX: 3. keep len(low) >= len(high)
            heapq.heappush(self.low, -heapq.heappop(self.high))

    def median(self):
        if len(self.low) > len(self.high):   # RETURN: odd count, the extra number sits in low
            return float(-self.low[0])
        return (-self.low[0] + self.high[0]) / 2    # RETURN: even count, the mean of the roots


mf, medians = MedianFinder(), []
for x in [5, 15, 1, 3]:
    mf.add(x)
    medians.append(mf.median())
print(medians)                               # [5.0, 10.0, 5.0, 4.0]

**Try it**
- After each `add`, print `sorted(-v for v in mf.low)` and `sorted(mf.high)`: `[5] []`, `[5] [15]`, `[1, 5] [15]`, `[1, 3] [5, 15]`. Every small number is at most every large one.
- Delete step 3 and rerun: the very first `median()` raises `IndexError`. Step 2 moved the 5 into `high`, and `low` is empty.
- On a fresh `MedianFinder()`, change `/ 2` to `// 2` and add 1, then 2: the median prints `1` instead of `1.5`.
- Predict the medians for `[5, 4, 3, 2, 1]` before running: `[5.0, 4.5, 4.0, 3.5, 3.0]`.

**Cooldown: heap + queue (Task Scheduler, Reorganize String).** Always run the ready task with the most copies left: it is the one that would otherwise force idle ticks at the end. A task that just ran waits n ticks on a FIFO "conveyor belt"; tasks leave the belt in the order they got on, so only its front needs checking. The release comes *after* the run: a task whose cooldown ends with this tick may run from the next tick on. Reorganize String is the same machine with a cooldown of one turn, so the belt shrinks to a single `held` letter, pushed back only after the next letter has been chosen.

In [ ]:
def least_interval(tasks, n):
    heap = [-c for c in Counter(tasks).values()]   # STATE: -copies left of the tasks ready now (max-heap)
    heapq.heapify(heap)
    cooling = deque()                        # STATE: (tick its cooldown ends, -copies left), oldest first
    time = 0                                 # INIT  (every task with copies left is in exactly one of the two)
    while heap or cooling:
        time += 1                            # one tick
        if heap:
            left = heapq.heappop(heap) + 1   # STEP: run the top (stored negated: +1 means one copy fewer)
            if left:                         # copies remain: it cools down for n ticks
                cooling.append((time + n, left))
        else:
            time = cooling[0][0]             # nothing is ready: skip the idle ticks at once
        if cooling and cooling[0][0] == time:     # FIX: the front's cooldown ends with this tick
            heapq.heappush(heap, cooling.popleft()[1])
    return time                              # RETURN


def reorganize(s):                           # 767: no two equal letters side by side
    heap = [(-c, ch) for ch, c in Counter(s).items()]   # STATE: max-heap of copies left
    heapq.heapify(heap)
    out, held = [], None                     # STATE: held = the letter just placed, sitting out one turn
    while heap:
        c, ch = heapq.heappop(heap)          # STEP: most copies left among the allowed letters
        out.append(ch)
        if held:
            heapq.heappush(heap, held)       # FIX: last turn's letter may be used again
        held = (c + 1, ch) if c + 1 < 0 else None
    return "" if held else "".join(out)      # RETURN: a letter still waiting would have to repeat


print(least_interval(["A", "A", "A", "B", "B", "B"], 2))   # 8   (A B _ A B _ A B)
print(least_interval(["A", "A", "A", "A", "B", "C"], 2))   # 10  (A B C A _ _ A _ _ A)
print(reorganize("aab"), repr(reorganize("aaab")), reorganize("aaabb"))   # aba '' ababa

**Try it**
- Change `(time + n, left)` to `(time + n - 1, left)`: the first call prints 6 instead of 8. Off by one on the ready time silently shrinks the cooldown to n − 1.
- Delete the `else` branch: the answers stay the same, but `least_interval(["A", "A"], 10**6)` now loops a million times; with the jump it returns 1000002 after three loops.
- Check the first call against the counting formula `(most - 1) * (n + 1) + ties` (`most` = the top count, `ties` = how many tasks have it): (3 − 1) × 3 + 2 = 8. The formula needs a `max(len(tasks), ...)` guard; the simulation does not.
- In `reorganize`, push the letter straight back instead of holding it (`heapq.heappush(heap, (c + 1, ch))` right after placing it, when copies remain): `"aab"` comes out as `"aab"`.

#### Stretch: hard heap patterns

The four patterns below are Hard problems built from the same two templates. Learn the core above first; come back here once those feel automatic.

**Greedy + heap: unlock, then take the best; or take now, regret later.** Sort by the key that *unlocks* options (capital, position, deadline), sweep, push each unlocked option into a heap keyed by its *value*, and pop only when you must choose, or when you must undo a choice. In the refuelling loop the FIX comes before the STEP: you may only take fuel from stations you have actually reached.

In [ ]:
def max_capital(k, w, profits, capital):                 # IPO (502)
    projects = sorted(zip(capital, profits))             # INIT: unlock order, cheapest first
    heap, i = [], 0                                      # STATE: max-heap (negated) of affordable profits
    for _ in range(k):
        while i < len(projects) and projects[i][0] <= w: # FIX: heap = every affordable, unused project
            heapq.heappush(heap, -projects[i][1])
            i += 1
        if not heap:                                     # nothing affordable, now or ever
            break
        w += -heapq.heappop(heap)                        # STEP: do the most profitable one
    return w                                             # RETURN


def min_refuel_stops(target, fuel, stations):            # (871) fuel = how far we can drive
    passed, stops = [], 0                                # STATE: max-heap (negated) of fuel we drove past
    for pos, gas in stations + [[target, 0]]:            # the target is the last "station"
        while fuel < pos:                                # FIX: stuck before pos, refuel retroactively
            if not passed:
                return -1                                # RETURN: nothing left to take
            fuel += -heapq.heappop(passed)               # the biggest station behind us
            stops += 1                                   # RECORD
        heapq.heappush(passed, -gas)                     # STEP: we reached pos, its fuel is an option
    return stops                                         # RETURN


def schedule_course(courses):                            # (630) courses = [duration, last_day]
    taken, time = [], 0                                  # STATE: max-heap (negated) of taken durations; time = their sum
    for duration, last_day in sorted(courses, key=lambda c: c[1]):   # INIT: by deadline
        heapq.heappush(taken, -duration)                 # STEP: take it now ...
        time += duration
        if time > last_day:                              # FIX: ... regret, drop the LONGEST course taken
            longest = -heapq.heappop(taken)
            time -= longest
    return len(taken)                                    # RETURN


print(max_capital(2, 0, [1, 2, 3], [0, 1, 1]))                                 # 4
print(min_refuel_stops(100, 10, [[10, 60], [20, 30], [30, 30], [60, 40]]))     # 2
print(schedule_course([[100, 200], [200, 1300], [1000, 1250], [2000, 3200]]))  # 3

**Try it**
- In `min_refuel_stops`, move `heapq.heappush(passed, -gas)` *above* the `while` loop and run `min_refuel_stops(100, 1, [[10, 100]])`: 1 instead of −1. The car refuelled at a station it never reached.
- In `schedule_course`, drop the *shortest* course instead (push `duration` without the minus, pop the root): `schedule_course([[5, 5], [4, 6], [2, 6]])` gives 1 instead of 2. Dropping the longest frees the most time.
- Remove the `if not heap: break` from `max_capital` and run `max_capital(1, 0, [5], [1])`: `IndexError`, since nothing is affordable with 0 capital.
- Print `-passed[0]` each time a stop is taken in the 100-mile example: 60, then 40. The decision is postponed until the car would run dry, and then it is easy.

**Lazy deletion (the skyline).** A heap can't remove an item from its middle. Lazy deletion says you don't have to: leave the dead entry in and throw it away when it reaches the top, because the top is the only thing you ever read. That is also why the FIX must run before the RECORD reads the top.

```text
 15           +-----------+              buildings [2,9,10] [3,7,15] [5,12,12]
 12           |           +--------------+
 10        +--+                          |        sweep the edges left to right;
  0  ------+                             +-----   live = max-heap of (height, right end)
           2  3           7              12       key point = where the top height changes
```

In [ ]:
def skyline(buildings):
    events = sorted([(l, -h, r) for l, r, h in buildings] +    # INIT: left edge (x, -height, right);
                    [(r, 0, 0) for l, r, h in buildings])      #   a right edge sorts after lefts at x
    live = [(0, math.inf)]                   # STATE: max-heap of (-height, right); the ground never ends
    points = []
    for x, neg_h, right in events:
        while live[0][1] <= x:               # FIX: the tallest has already ended, delete it now
            heapq.heappop(live)
        if neg_h:                            # STEP: a left edge, this building goes live
            heapq.heappush(live, (neg_h, right))
        height = -live[0][0]
        if not points or points[-1][1] != height:   # RECORD: the outline changes here
            points.append([x, height])
    return points                            # RETURN


print(skyline([[2, 9, 10], [3, 7, 15], [5, 12, 12], [15, 20, 10], [19, 24, 8]]))
# [[2, 10], [3, 15], [7, 12], [12, 0], [15, 10], [20, 8], [24, 0]]

**Try it**
- Change `<=` to `<` in the `while` and run `skyline([[0, 2, 3], [2, 4, 1]])`: `[[0, 3], [4, 1]]` instead of `[[0, 3], [2, 1], [4, 0]]`. A building that ends at x does not cover x.
- Remove the `points[-1][1] != height` test (append on every event): the output gains repeats like `[5, 15]` and `[9, 12]`, points where the height did not change.
- Print `x, sorted(live)` right after the `while`: at x = 9 the building `[2, 9, 10]` has ended, yet `(-10, 9)` is still in the heap under the taller `(-12, 12)`. It is harmless there, and it is popped at x = 12 when it surfaces.

**Lowest frontier first (Trapping Rain Water II).** Water in a cell can rise only as high as the lowest wall on its best escape route to the border. Start with the border as the wall, always breach the wall at its *lowest* cell, and let the neighbour behind it fill up to that level and join the wall. It is Dijkstra with `max` in place of `+` ([Graphs III](09_Graphs.ipynb#s19)).

```text
heights            level the water reaches     trapped = level - height
1 4 3 1 3 2        1 4 3 1 3 2                 . . . . . .
3 2 1 3 2 4   ->   3 3 3 3 3 4          ->     . 1 2 0 1 .      total 4
2 3 3 2 3 1        2 3 3 2 3 1                 . . . . . .
```

In [ ]:
def trap_rain_water(height):
    m, n = len(height), len(height[0])
    wall = [(height[r][c], r, c) for r in range(m) for c in range(n)
            if r in (0, m - 1) or c in (0, n - 1)]   # STATE + INIT: (level, r, c) of the wall, the border first
    heapq.heapify(wall)
    seen = {(r, c) for _, r, c in wall}      # STATE: cells already in the wall (or retired)
    water = 0
    while wall:
        level, r, c = heapq.heappop(wall)    # STEP: the wall's lowest cell, water escapes here first
        for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
            if 0 <= nr < m and 0 <= nc < n and (nr, nc) not in seen:
                seen.add((nr, nc))           # mark when pushing, as in BFS
                water += max(0, level - height[nr][nc])                    # RECORD: it fills up to level
                heapq.heappush(wall, (max(level, height[nr][nc]), nr, nc)) # FIX: then it joins the wall
    return water                             # RETURN


print(trap_rain_water([[1, 4, 3, 1, 3, 2], [3, 2, 1, 3, 2, 4], [2, 3, 3, 2, 3, 1]]))   # 4
print(trap_rain_water([[3, 3, 3, 3, 3], [3, 2, 2, 2, 3], [3, 2, 1, 2, 3],
                       [3, 2, 2, 2, 3], [3, 3, 3, 3, 3]]))                          # 10

**Try it**
- Push the neighbour's own height instead of `max(level, height[nr][nc])`: the second grid gives 2 instead of 10. A cell that just filled up rejoins the wall at its low floor height, so its neighbours are filled from that "hole" instead of from the real wall.
- Replace the heap with a FIFO queue (`deque`, `popleft`, `append`): the first grid gives 5 instead of 4. Without "lowest first", a cell gets filled from a wall that is not its real bottleneck.
- Print `level, r, c` for each pop on the first grid: the levels never go down. That monotone order is what makes each cell's level final when it is first reached.
- A grid with fewer than 3 rows or columns has no inside: `trap_rain_water([[5, 1, 5]])` is 0.

**Merge + running max (smallest range covering k lists).** Put one finger on each sorted list. The fingered values cover every list, and they span `[min, max]`. The only way to shrink that range is to raise the min, and only the finger *on* the min can do it: advance it, update the running max, and stop when some list runs out.

In [ ]:
def smallest_range(lists):
    heap = [(lst[0], i, 0) for i, lst in enumerate(lists)]   # STATE + INIT: one finger (value, list, index) per list
    heapq.heapify(heap)
    hi = max(lst[0] for lst in lists)        # STATE: the rightmost finger (a running max)
    best = [-math.inf, math.inf]
    while True:
        lo, i, j = heapq.heappop(heap)       # STEP: the leftmost finger
        if hi - lo < best[1] - best[0]:      # RECORD before moving it (< keeps the earliest start)
            best = [lo, hi]
        if j + 1 == len(lists[i]):           # list i is used up: no later range covers it
            return best                      # RETURN
        nxt = lists[i][j + 1]
        hi = max(hi, nxt)
        heapq.heappush(heap, (nxt, i, j + 1))    # FIX: list i gets its next finger


print(smallest_range([[4, 10, 15, 24, 26], [0, 9, 12, 20], [5, 18, 22, 30]]))   # [20, 24]
print(smallest_range([[1, 2, 3], [1, 2, 3], [1, 2, 3]]))                        # [1, 1]

**Try it**
- Move the "used up" check above the RECORD lines and run `smallest_range([[1], [2], [3]])`: `[-inf, inf]` instead of `[1, 3]`. The last range was never recorded.
- Change `<` to `<=` and run `smallest_range([[1, 3], [2, 4]])`: `[3, 4]` instead of `[1, 2]`. Equal widths must keep the earliest start.
- Delete `hi = max(hi, nxt)`: the first example returns `[20, 5]`, an upside-down "range". `hi` froze at the first fingers' maximum; the right end must follow every new finger.

### Say it in the interview

> "Sorting everything costs O(n log n) and orders items I'm going to throw away. I only need the k best, so I keep a min-heap capped at k: its root is the weakest of my k, and every newcomer only has to beat that root. Anything that leaves was beaten by k numbers still in the heap, so it can't be in the top k. Each step is O(log k): O(n log k) time, O(k) space, and it works on a stream. For k sorted lists I keep one head per list: every pop is the next item overall, O(N log k)."

While coding, point at the `if len(heap) > k` line and say the invariant: "after this line the heap holds exactly the k largest so far, and `heap[0]` is the k-th largest". In a merge, point at the refill right after the pop: "the list I just took from gets its next item in, so every list always has its head in the heap". And say why the tuple has an index in the middle: "so ties never compare the payload". Likely follow-ups and your answers:

- *Faster?* → quickselect: O(n) on average, but it needs all the data in memory ([Sorting & Selection](05_Binary_Search_Sorting.ipynb#s23)).
- *k is close to n?* → keep the n − k smallest in a max-heap instead, or heapify everything and pop k times: O(n + k log n).
- *Sorted output?* → pop the k items: O(k log k).
- *Ties or a custom order?* → tuple keys, or a class with `__lt__`.
- *Priorities change?* → push a fresh entry and skip stale ones when they are popped.
- *The data is spread over many machines?* → top k on each machine, then a k-way merge of those lists.

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| Course Schedule III | `heap/course_schedule_iii.py` | sort by deadline, take every course; past the deadline, drop the longest taken (max-heap) |
| Design Twitter | `heap/design_twitter.py` | each user's tweets are already sorted by time: the feed is a k-way merge, stopped after 10 pops |
| Find Median from Data Stream | `heap/find_median_from_data_stream.py` · `practice/simple/34_find_median_from_data_stream.py` | max-heap of the low half, min-heap of the high half, sizes within 1; the median sits at the roots |
| IPO | `heap/ipo.py` | sort by capital; push newly affordable profits into a max-heap; take the top k times |
| K Closest Points to Origin | `heap/k_closest_points_to_origin.py` · `practice/simple/32_k_closest_points_to_origin.py` | max-heap capped at k on squared distance; the root is the farthest point kept |
| Kth Largest Element in a Stream | `heap/kth_largest_element_in_a_stream.py` | min-heap capped at k; its root is the k-th largest after every add |
| Kth Largest Element in an Array | `heap/kth_largest_element_in_an_array.py` | min-heap capped at k for O(n log k), or quickselect toward index n − k for O(n) average |
| K-th Smallest Prime Fraction | `heap/kth_smallest_prime_fraction.py` | each numerator is a sorted row of fractions: k-way merge, pop k − 1 times |
| Maximum Performance of a Team | `heap/maximum_performance_of_a_team.py` | sort by efficiency, high first (it is the current minimum); min-heap of the k best speeds with a running sum |
| Meeting Rooms III | `heap/meeting_rooms_iii.py` | two heaps: free room ids, busy (end, room); free ended rooms first; a delayed meeting keeps its length |
| Merge k Sorted Lists | `heap/merge_k_sorted_lists.py` · `practice/simple/35_merge_k_sorted_lists.py` | heap of the k heads as (val, i, node); pop the smallest, push its next |
| Minimize Deviation in Array | `heap/minimize_deviation_in_array.py` | double every odd so only halving is left; halve the max (max-heap) and track the min; stop at an odd max |
| Minimum Number of Refueling Stops | `heap/minimum_number_of_refueling_stops.py` | drive past stations into a max-heap; when stuck, retroactively take the biggest |
| Rearrange String k Distance Apart | `heap/rearrange_string_k_distance_apart.py` | max-heap by count + FIFO cooldown queue of length k; an empty heap mid-way means impossible |
| Reorganize String | `heap/reorganize_string.py` | possible iff top count ≤ (n + 1) // 2; place the most frequent letter, holding the last one out a turn |
| Smallest Range Covering Elements from K Lists | `heap/smallest_range_covering_elements_from_k_lists.py` | one finger per list: heap gives the min, a running max the max; advance the min until a list runs out |
| Task Scheduler | `heap/task_scheduler.py` · `practice/simple/33_task_scheduler.py` | run the ready task with most copies left (max-heap); park it in a FIFO cooldown queue |
| The Skyline Problem | `heap/the_skyline_problem.py` | sweep the edges; max-heap of live (height, right) with lazy deletion; emit when the top height changes |
| Top K Frequent Words | `heap/top_k_frequent_words.py` | size-k heap whose root is the worst word (`__lt__`: fewer copies, or later word); or heapify (−count, word), pop k |
| Trapping Rain Water II | `heap/trapping_rain_water_ii.py` | the border is a wall; breach its lowest cell (min-heap); neighbours fill to that level and join the wall |

### Self-check

1. You want the 10 largest numbers of a stream of a billion. Which heap, which size, and what does its root mean?
<details><summary>Answer</summary>A min-heap capped at 10. Its root is the smallest of the 10 largest seen so far (the 10th largest), so a new number stays only if it beats the root, and then the root is popped. O(log 10) per number and O(10) memory.</details>

2. Why does the keep-k template push first and evict second?
<details><summary>Answer</summary>After the push the heap holds k + 1 candidates, and popping removes the weakest of all of them, which may be the newcomer itself. Evicting first removes a kept item before the newcomer has been compared with anything: <code>k_largest([5, 1], 1)</code> would keep 1.</details>

3. `heapq.heappush(heap, (dist, node))` works on small tests and then crashes with `TypeError`. Why, and what is the fix?
<details><summary>Answer</summary>When two distances are equal, Python compares the next tuple field, the nodes, and they don't support <code>&lt;</code>. Put a unique tie-breaker before the payload: <code>(dist, i, node)</code> with a list index or a running counter.</details>

4. In the median finder, why does every number go into `low` first and then move `low`'s maximum to `high`?
<details><summary>Answer</summary>That two-step dance keeps the ordering rule without any comparisons of your own: whatever crosses the seam is the largest of the small half (including the new number), so every number in <code>low</code> stays at most every number in <code>high</code>. Step 3 then fixes only the sizes.</details>

<a id="s14"></a>

## Intervals & Sweep Line

> Lay the intervals on a number line and sweep a vertical line from left to right. Sort once, and every decision only needs what the line is touching right now: the block being merged, the room that frees up first, or a running count of open intervals.

**Reach for it when** the input is pairs `[start, end]` (meetings, bookings, ranges, buildings, rectangles) and the question is about overlap: merge them, insert one, intersect two lists of them, keep the most / remove the fewest so that none overlap, find how many are open at once (rooms), find the free gaps, or answer "which intervals contain point q?".

**In this repo:** `intervals/` (8 problems) · bank: `practice/simple/48_merge_intervals.py`, `practice/simple/49_meeting_rooms_ii.py` · basics: `practice/simple/basics/sorting/06_python_sort_keys_and_stability.py` (sort keys such as `(end, -start)`), `practice/simple/basics/heaps/` (the min-heap behind meeting rooms) · sweeps that need a heap live in the Heaps section (skyline, Meeting Rooms III); covering a range and "two points per interval" live in the Greedy section; [Two Pointers](03_Two_Pointers_Sliding_Window_Strings.ipynb#s05) walks two sorted lists the way Interval List Intersections does below.

### The picture

```text
0   2   4   6   8   10  12  14  16  18
  [===]                                     [1,3]
    [=======]                               [2,6]    2 <= 3: same block, its end stretches to 6
                [===]                       [8,10]   8 > 6: block [1,6] is final, open a new one
                  [=====]                   [9,12]   9 <= 10: stretch to 12
                              [=====]       [15,18]  15 > 12: block [8,12] is final
  [=========]   [=======]     [=====]       merged:  [1,6] [8,12] [15,18]
```

**Why it is fast:** the brute force compares every pair, O(n²), and merging until nothing changes is O(n³). After sorting by start, an interval can only overlap the block you are building *right now*: every later interval starts even later, so once one starts past the block's end, that block is final and never looked at again. One sort and one sweep: O(n log n).

### Overlap in one line

```text
two intervals overlap  <=>  each one starts before the other one ends

  a [=======)              a [=======)                a [=====)
  b     [=======)          b         [=======)        b           [=====)
    overlap                  touching: a.end == b.start   apart

half-open [s, e)   a.start <  b.end and b.start <  a.end     meetings, bookings
closed    [s, e]   a.start <= b.end and b.start <= a.end     merge intervals, "touching overlaps"
```

It is easier to say when two intervals do *not* overlap: one ends before the other starts. Negate that and you get the one-liner above, which handles partial, nested and identical intervals with no case analysis. When they do overlap, the common part is `[max(a.start, b.start), min(a.end, b.end)]`.

The only real question is what happens at a shared endpoint, and the problem statement decides it:

| Problem | Ends | `[1, 2]` and `[2, 3]` | So the test is |
|---|---|---|---|
| Merge Intervals (56), Insert Interval (57) | closed | overlap: one block `[1, 3]` | `start <= block_end` |
| Non-overlapping Intervals (435) | treated as half-open | no overlap: keep both | keep if `start >= kept_end` |
| Meeting Rooms II (253), My Calendar (729, 732) | half-open `[s, e)` | no overlap: one room is enough | room is free if `end <= start` |
| Interval List Intersections (986) | closed | they share the point 2 | meet if `max(starts) <= min(ends)` |
| Employee Free Time (759) | busy blocks | no free time between them | gap only if `start > busy_until` |

### From idea to code

**The idea in one sentence:** *sort by start, then sweep: to merge, keep one "current" block that each interval either stretches or closes; to count, keep the end times of what is still running, free what has ended, then add the newcomer.*

| Decision | Merge (56, 57) | How many at once (253) |
|---|---|---|
| **State / Definition** | `merged[-1]` = the block being built (the union of everything so far that chains into it); `merged[:-1]` = finished blocks | `ends` = min-heap of the end times of the meetings still running at `start` |
| **Invariant** | `merged` is sorted and disjoint and covers exactly the intervals seen so far | after the fix, every end in the heap is `> start` |
| **Step** | the interval stretches the block (`max` of the ends) or opens a new block | push the new meeting's end: it takes a room |
| **Fix** | none: after sorting, only `merged[-1]` can be touched | `while ends[0] <= start: pop` (those rooms are free again) |
| **Record** | a block is final when an interval starts after its end (or the input ends) | `rooms = max(rooms, len(ends))` after the push |
| **Init** | sort by start; `merged = []` | sort by start; `ends = []` |
| **Return** | `merged` (empty input gives `[]`) | `rooms` |

The same idea, sentence by sentence:

| In words | In code |
|---|---|
| "sort by start" | `sorted(intervals)` (lists compare by start, then by end) |
| "the block I am building" | `merged[-1]` |
| "it starts inside the block (touching counts)" | `start <= merged[-1][1]` |
| "stretch the block" | `merged[-1][1] = max(merged[-1][1], end)` |
| "a gap: open a new block" | `merged.append([start, end])` |
| "do a and b overlap?" (half-open) | `a[0] < b[1] and b[0] < a[1]` |
| "their common part" | `[max(a[0], b[0]), min(a[1], b[1])]` |
| "the room that frees up first" | `ends[0]` of a min-heap of end times |
| "how many are open right now" | `len(ends)`, or a running sum of +1 at each start and −1 at each end |

**Free, then occupy, then count.** The RECORD reads how many meetings run at this instant, so it must come after both other lines: the ended meetings must be gone (move the freeing loop below the count, and the back-to-back `[1, 5]`, `[5, 9]` need 2 rooms) and the newcomer must be in (move the count above the push, and `[[1, 5], [2, 6], [3, 7]]` needs only 2 rooms, not 3). Freeing before the push reads naturally: the newcomer may take a room whose meeting ended exactly at its start.

In [ ]:
def merge(intervals):
    merged = []                                       # STATE: finished blocks + the current one, merged[-1]
    for start, end in sorted(intervals):              # INIT: sort by start (ties by end)
        if merged and start <= merged[-1][1]:         # starts inside the current block:
            merged[-1][1] = max(merged[-1][1], end)   # STEP: stretch it (max: it may be nested)
        else:
            merged.append([start, end])               # STEP + RECORD: the block before is final; open a new one
    return merged                                     # RETURN


def overlaps(a, b):                                   # half-open [s, e): touching is NOT overlapping
    return a[0] < b[1] and b[0] < a[1]


def min_rooms_heap(meetings):
    ends, rooms = [], 0                      # STATE: min-heap of end times of the meetings running now
    for start, end in sorted(meetings):      # INIT: by start
        while ends and ends[0] <= start:     # FIX: free every room whose meeting is over (half-open: <=)
            heapq.heappop(ends)
        heapq.heappush(ends, end)            # STEP: this meeting takes a room
        rooms = max(rooms, len(ends))        # RECORD: rooms in use right now
    return rooms                             # RETURN


print(merge([[1, 3], [2, 6], [8, 10], [15, 18]]))     # [[1, 6], [8, 10], [15, 18]]
print(merge([[1, 4], [4, 5]]), overlaps([1, 4], [4, 5]), overlaps([1, 9], [3, 4]))   # [[1, 5]] False True
print(min_rooms_heap([[0, 30], [5, 10], [15, 20]]), min_rooms_heap([[1, 5], [5, 9]]))  # 2 1

**Try it**
- Change `max(merged[-1][1], end)` to plain `end` and run `merge([[1, 10], [2, 3], [4, 5]])`: `[[1, 3], [4, 5]]`. The nested `[2, 3]` shrank the block.
- Change `<=` to `<` in `merge`: `merge([[1, 4], [4, 5]])` now gives `[[1, 4], [4, 5]]`. Closed or half-open is a decision you read from the problem, then write once.
- Replace `sorted(intervals)` with `intervals` and run `merge([[8, 10], [1, 3], [2, 6]])`: `[[8, 10]]`. The test only looks at the block's end, so `[1, 3]` and `[2, 6]` were swallowed by a block they never touch.
- In `min_rooms_heap`, move the `while` loop below the RECORD line: `min_rooms_heap([[1, 5], [5, 9]])` needs 2 rooms. The room was freed one step too late.

**Why the peak is the answer.** At the busiest instant that many meetings run at once, and each needs its own room, so fewer rooms is impossible. That many is also enough: hand out rooms in start order, each to a room whose meeting has ended. One is always free, because otherwise one more meeting than the peak would be running at that moment.

### Watch it work

Fed in a shuffled order, the sweep still sees the intervals sorted:

In [ ]:
def trace_merge(intervals):
    merged = []
    for start, end in sorted(intervals):
        if merged and start <= merged[-1][1]:
            old = merged[-1][1]
            merged[-1][1] = max(old, end)
            note = f"{start} <= {old}: stretch to {merged[-1][1]}"
        else:
            note = f"{start} > {merged[-1][1]}: {merged[-1]} is final" if merged else "open the first block"
            merged.append([start, end])
        print(f"[{start:>2},{end:>3}]  {note:<26} merged={merged}")


trace_merge([[8, 10], [1, 3], [15, 18], [2, 6], [9, 12]])

**Try it**
- Run `trace_merge([[1, 10], [2, 3], [4, 5]])`: the block stays `[1, 10]` while the small intervals are swallowed. That is the `max` at work.
- Run `trace_merge([[1, 2], [2, 3], [3, 4]])`: each touches the next, so the chain becomes one block `[1, 4]`.
- Run `trace_merge([[1, 2], [3, 4], [2, 3]])`: one block `[1, 4]`. Without the sort the loop gives `[[1, 2], [3, 4]]`: a late interval can bridge two blocks that were already closed.

### Where it goes wrong

1. **Sorting by the wrong key, or not at all.** Merging and rooms: sort by start. "Keep the most / remove the fewest": sort by **end**. With `[[1, 100], [2, 3], [4, 5]]`, keeping greedily in start order keeps `[1, 100]` and removes 2; end order removes only 1.
2. **`end` instead of `max(...)`** when stretching: a nested interval shrinks the block (`[[1, 10], [2, 3], [4, 5]]` → `[[1, 3], [4, 5]]`).
3. **Comparing with the previous interval instead of the block.** After the nested `[2, 3]`, the previous end (3) is below the block's end (10), so `[[1, 10], [2, 3], [4, 5]]` becomes `[[1, 10], [4, 5]]`. Always compare with `merged[-1]`.
4. **`<` vs `<=` at a shared endpoint.** Closed ends (merge): `[1, 4]` and `[4, 5]` overlap. Half-open (meetings): a meeting ending at 10 frees its room for one starting at 10.
5. **Editing the caller's lists.** `merged.append(iv)` followed by `merged[-1][1] = ...` changes the input in place. Append a fresh `[start, end]`.
6. **Offline answers in the wrong order.** If you sort the queries to sweep them, store each answer under its original query, then rebuild the original order.
7. **The +1/−1 order at equal times.** For half-open intervals the −1 must come before the +1 at the same time, or back-to-back meetings count as overlapping. Sorting `(time, delta)` tuples does it for free: −1 sorts first.
8. **Adding a strip with the new set.** In a 2-D sweep, add the strip to the left of x with the *old* active set, then apply the event at x.

### Edge cases to say out loud

Empty list · one interval · touching ends (closed vs half-open) · nested intervals · identical intervals · unsorted input · zero-length intervals `[5, 5]` · negative coordinates · inserting before all, after all, or around all.

In [ ]:
assert merge([]) == []
assert merge([[5, 5]]) == [[5, 5]]                          # zero length is still an interval
assert merge([[1, 10], [2, 3]]) == [[1, 10]]                # nested: max keeps the 10
assert merge([[8, 9], [1, 3], [2, 4]]) == [[1, 4], [8, 9]]  # unsorted input
assert merge([[1, 2], [1, 2]]) == [[1, 2]]                  # identical
assert merge([[-5, -1], [-2, 0]]) == [[-5, 0]]
assert not overlaps([1, 2], [2, 3]) and overlaps([1, 3], [2, 2.5])
assert min_rooms_heap([]) == 0 and min_rooms_heap([[1, 2]]) == 1
ivs = [[1, 3], [2, 4]]
merge(ivs)
assert ivs == [[1, 3], [2, 4]]                              # the caller's lists are untouched
print("edge cases pass")

**Try it**
- Predict `merge([[1, 4], [0, 0]])` before running it: `[[0, 0], [1, 4]]` (they don't touch).
- Rewrite the loop as `for iv in sorted(intervals):` with `merged.append(iv)` and `iv[0]`, `iv[1]` inside: the last assert fails, because the caller's `[1, 3]` became `[1, 4]`. `sorted` copies the outer list, not the inner ones.
- What should `min_rooms_heap([[1, 3], [1, 3], [1, 3]])` return? Write the assert first (3).

### Variations

| Variation | What changes from the template | Problems |
|---|---|---|
| **Merge** | the template | 56 |
| **Insert into a sorted, disjoint list** | no sort: copy what ends before, swallow what overlaps, copy the rest | 57 |
| **Keep the most / remove the fewest** | sort by END; keep an interval if it starts at or after the last kept end | 435, 452 |
| **How many at once** | the rooms template, or +1/−1 events with ends first at equal times; Car Pooling compares the peak with a capacity | 253, 1094 |
| **Intersect two sorted lists** | two pointers; the common part is `[max starts, min ends]`; advance whichever ends first | 986 |
| **Free gaps** | merge everyone's busy time; the holes between blocks are free | 759 |
| **Bookings one at a time** | keep accepted bookings sorted; check only the two neighbours with `bisect` | 729 |
| **Covered intervals** | sort by `(start, -end)`; an interval is covered iff its end ≤ the largest end so far | 1288 |
| **Cover a range with the fewest intervals** | jump-game reach (Greedy section) | 1326 |
| **At least two points in every interval** | sort by end, place points at the right end (Greedy section) | 757 |
| **Rooms with ids, delayed meetings** | two heaps: free ids, busy ends (Heaps section) | 2402 |
| *Stretch:* **point queries** | sort the queries offline; push intervals that have started; lazily pop those that ended | 1851 |
| *Stretch:* **running bookings** | difference map: +1 at start, −1 at end; sweep the sorted keys | 731, 732 |
| *Stretch:* **area of a union** | sweep x; each strip adds width × union length of the active y-intervals | 850 |

**Insert into a sorted list.** The list is already sorted and disjoint, so no sort is needed: the intervals that touch the new one form a single run. Copy everything before the run, swallow the run into the new interval, copy everything after.

```text
[1,2]   [3,5]   [6,7]   [8,10]   [12,16]          new = [4,8]
copy    swallow swallow swallow  copy             ->  [1,2] [3,10] [12,16]
```

In [ ]:
def insert(intervals, new):
    out, i, n = [], 0, len(intervals)        # STATE: out = the answer so far; i = next interval to look at
    start, end = new                         # STATE: the new interval, growing as it swallows others
    while i < n and intervals[i][1] < start:     # 1. ends before new starts: copy it untouched
        out.append(intervals[i])
        i += 1
    while i < n and intervals[i][0] <= end:      # 2. overlaps (touching counts):
        start = min(start, intervals[i][0])      # STEP: swallow it
        end = max(end, intervals[i][1])
        i += 1
    out.append([start, end])                     # RECORD: the grown new interval
    return out + intervals[i:]                   # RETURN: 3. the rest starts after new ends


print(insert([[1, 3], [6, 9]], [2, 5]))                              # [[1, 5], [6, 9]]
print(insert([[1, 2], [3, 5], [6, 7], [8, 10], [12, 16]], [4, 8]))   # [[1, 2], [3, 10], [12, 16]]

**Try it**
- Insert at the very end, `insert([[1, 2]], [5, 6])`, and at the very front, `insert([[3, 4]], [1, 2])`: phase 2 is empty both times, and the `append` between the loops still lands in the right place.
- Change phase 1's `<` to `<=` and run `insert([[1, 2]], [2, 3])`: `[[1, 2], [2, 3]]` instead of `[[1, 3]]`. The touching interval was copied instead of swallowed.
- `insert([], [4, 8])` gives `[[4, 8]]`: both loops are skipped. This is O(n) while `merge` is O(n log n); point at the line that is missing.

**Keep the most: sort by end.** To fit the most intervals, always keep the one that *ends* first: it leaves the most room for everything after it. Any best answer can swap its first interval for the earliest-ending one without creating a clash (an exchange argument), so the greedy never loses. Every interval that starts before the last kept end is a removal.

In [ ]:
def erase_overlap_intervals(intervals):
    kept_end, removed = -math.inf, 0         # STATE: kept_end = end of the last kept interval
    for start, end in sorted(intervals, key=lambda iv: iv[1]):   # INIT: earliest END first
        if start >= kept_end:                # fits after the last kept one (touching is fine)
            kept_end = end                   # STEP: keep it
        else:
            removed += 1                     # RECORD: it clashes with a kept one that ends no later
    return removed                           # RETURN


print(erase_overlap_intervals([[1, 2], [2, 3], [3, 4], [1, 3]]))   # 1
print(erase_overlap_intervals([[1, 2], [1, 2], [1, 2]]))           # 2
print(erase_overlap_intervals([[1, 100], [2, 3], [4, 5]]))         # 1

**Try it**
- Sort by start instead (`key=lambda iv: iv[0]`) and rerun the last call: 2 instead of 1. `[1, 100]` is kept first and blocks everything.
- Sort by length (`key=lambda iv: iv[1] - iv[0]`) and run `erase_overlap_intervals([[1, 5], [4, 6], [5, 10]])`: 2 instead of 1. The short `[4, 6]` clashes with both of the others.
- Change `>=` to `>`: `erase_overlap_intervals([[1, 2], [2, 3]])` now removes 1 instead of 0, as if touching intervals overlapped.
- The same loop counts arrows for Minimum Arrows (452), where balloons are closed intervals: sort by end and shoot a new arrow only when `start > arrow`. `[[10, 16], [2, 8], [1, 6], [7, 12]]` needs 2 arrows, and so does `[[1, 2], [2, 3], [3, 4], [4, 5]]`.

**How many at once, as a +1/−1 sweep.** The room count is the largest number of meetings running at the same instant, so you can also turn each meeting into two events, +1 at its start and −1 at its end, and keep a running total. The heap version knows *which* room frees up next (Meeting Rooms III builds on it); the sweep is shorter and also works when bookings arrive one at a time (My Calendar III, below). LeetCode 253 promises `start < end`; with zero-length meetings the two disagree (`[[5, 5], [5, 5]]`: heap 1, sweep 0).

```text
meetings [0,30) [5,10) [15,20)
time      0    5   10   15   20   30
event    +1   +1   -1   +1   -1   -1
open      1    2    1    2    1    0          peak = 2 rooms
```

In [ ]:
def min_rooms_sweep(meetings):
    events = sorted([(s, 1) for s, e in meetings] + [(e, -1) for s, e in meetings])   # INIT: (t, -1) before (t, 1)
    open_now = rooms = 0                     # STATE: open_now = meetings running after this event
    for _, delta in events:
        open_now += delta                    # STEP
        rooms = max(rooms, open_now)         # RECORD
    return rooms                             # RETURN


print(min_rooms_sweep([[0, 30], [5, 10], [15, 20]]), min_rooms_sweep([[1, 5], [5, 9]]))   # 2 1

**Try it**
- Sort on time only, `sorted(..., key=lambda ev: ev[0])`: `[[1, 5], [5, 9]]` needs 2 rooms. Python's sort is stable, so at time 5 the start (listed first) is now processed before the end.
- Print `events` for the first call and run the totals by hand: 1, 2, 1, 2, 1, 0.
- Compare with the heap on random meetings (`start < end`): the two always agree, because both measure the largest number of meetings open at once.

**Intersect two sorted lists (Interval List Intersections).** Both lists are sorted and disjoint, so walk them with two pointers. The current pair meets on `[max of the starts, min of the ends]` when that is non-empty. Then drop the interval that ends first: it can't meet anything later in the other list, because everything there starts even later.

In [ ]:
def interval_intersection(A, B):             # 986: both sorted and disjoint, closed ends
    i = j = 0                                # STATE: the current interval of each list
    out = []
    while i < len(A) and j < len(B):
        lo = max(A[i][0], B[j][0])           # the common part starts at the later start
        hi = min(A[i][1], B[j][1])           # ... and ends at the earlier end
        if lo <= hi:                         # RECORD: they meet (touching counts)
            out.append([lo, hi])
        if A[i][1] < B[j][1]:                # STEP: the one that ends first can't meet anything later
            i += 1
        else:
            j += 1
    return out                               # RETURN


print(interval_intersection([[0, 2], [5, 10], [13, 23], [24, 25]], [[1, 5], [8, 12], [15, 24], [25, 26]]))
# [[1, 2], [5, 5], [8, 10], [15, 23], [24, 24], [25, 25]]

**Try it**
- Change `lo <= hi` to `lo < hi`: `[[1, 2], [8, 10], [15, 23]]`. The single-point intersections `[5, 5]`, `[24, 24]`, `[25, 25]` are lost.
- Advance the pointer that ends *later* (swap `i += 1` and `j += 1`): only `[[1, 2]]` is found. You threw away the interval that could still meet the next one.
- Print `i, j` at the top of the loop: each step moves exactly one pointer, so the loop runs at most `len(A) + len(B)` times.

**Free gaps (Employee Free Time).** Free time for everyone is what is left after the union of everyone's busy time. Merge all the busy intervals; the holes between consecutive blocks are the answer.

In [ ]:
def employee_free_time(schedules):
    busy = merge([iv for person in schedules for iv in person])   # STATE: everyone's busy time, merged
    return [[a[1], b[0]] for a, b in zip(busy, busy[1:])]         # RETURN: the holes between blocks


print(employee_free_time([[[1, 2], [5, 6]], [[1, 3]], [[4, 10]]]))          # [[3, 4]]
print(employee_free_time([[[1, 3], [6, 7]], [[2, 4]], [[2, 5], [9, 12]]]))  # [[5, 6], [7, 9]]

**Try it**
- Add a person who works `[[3, 4]]` to the first call: the answer becomes `[]`. Touching blocks merge (`<=`), so no zero-length gap can appear.
- Print `busy` for the second call: `[[1, 5], [6, 7], [9, 12]]`, and read the gaps straight off it.
- Each employee's list is already sorted, so rewrite it as a heap k-way merge (one head per employee, like `merge_sorted` in the Heaps section): pop the earliest start, record a gap if it starts after the latest end so far, push that employee's next interval. It prints the same answers in O(N log k) instead of O(N log N). (LeetCode 759 passes `Interval` objects: read `.start` and `.end`.)

**Bookings one at a time (My Calendar I).** Each booking must not overlap any accepted one. Keep the accepted bookings sorted; then only the two neighbours of the new booking can overlap it: the one just before (does it end after `start`?) and the one just after (does it start before `end`?).

In [ ]:
class MyCalendar:                                # 729
    def __init__(self):
        self.starts, self.ends = [], []          # STATE: accepted bookings, sorted and disjoint

    def book(self, start, end):                  # half-open [start, end)
        i = bisect.bisect_right(self.starts, start)          # bookings 0..i-1 start at or before `start`
        if (i > 0 and self.ends[i - 1] > start) or (i < len(self.starts) and self.starts[i] < end):
            return False                         # RETURN: it overlaps a neighbour
        self.starts.insert(i, start)             # STEP: keep both lists sorted
        self.ends.insert(i, end)
        return True                              # RETURN


cal1 = MyCalendar()
print([cal1.book(s, e) for s, e in [(10, 20), (15, 25), (20, 30)]])   # [True, False, True]

**Try it**
- Then book `(5, 10)`: `True`. It ends exactly where `(10, 20)` starts, and the ends are half-open.
- Change `self.ends[i - 1] > start` to `>=`: the third booking `(20, 30)` is now rejected (`[True, False, False]`). Touching bookings clash.
- Check only the booking before (drop the second condition): on a fresh calendar holding `(20, 30)`, `book(15, 25)` is accepted, although they overlap.

#### Stretch: hard interval patterns

The three patterns below are Hard problems: offline queries, a difference map, and a 2-D sweep. They reuse the sweep and the heap from the core; learn those first.

**Point queries (Minimum Interval to Include Each Query).** Answer the queries *offline*, in increasing order. As the query point moves right, intervals that have *started* (`left <= q`) join a min-heap keyed by size; intervals that have *ended* (`right < q`) are dead for this and every later query, so pop them lazily when they reach the top.

```text
intervals sorted by left: [1,4] [2,4] [3,6] [4,4]          (sizes 4, 3, 4, 1)
q = 2: push [1,4] [2,4]                          top [2,4]  -> 3
q = 3: push [3,6]                                top [2,4]  -> 3
q = 4: push [4,4]                                top [4,4]  -> 1
q = 5: pop [4,4], [2,4], [1,4] (they ended)      top [3,6]  -> 4
```

In [ ]:
def min_interval(intervals, queries):
    intervals = sorted(intervals)                # INIT: by left end
    heap, i, best = [], 0, {}                    # STATE: heap = (size, right) of intervals that have started
    for q in sorted(set(queries)):               # offline: queries in increasing order
        while i < len(intervals) and intervals[i][0] <= q:     # STEP: every interval started by q joins
            left, right = intervals[i]
            heapq.heappush(heap, (right - left + 1, right))
            i += 1
        while heap and heap[0][1] < q:           # FIX: lazy deletion, the top ended before q
            heapq.heappop(heap)
        best[q] = heap[0][0] if heap else -1     # RECORD
    return [best[q] for q in queries]            # RETURN: back in the original order


print(min_interval([[1, 4], [2, 4], [3, 6], [4, 4]], [2, 3, 4, 5]))      # [3, 3, 1, 4]
print(min_interval([[2, 3], [2, 5], [1, 8], [20, 25]], [2, 19, 5, 22]))  # [2, -1, 4, 6]

**Try it**
- Change `heap[0][1] < q` to `<=`: the first call gives `[3, 3, 4, 4]`. A query equal to an interval's right end is inside it, so `[4, 4]` must answer q = 4.
- Return `[best[q] for q in sorted(queries)]` instead: the second call gives `[2, 4, -1, 6]`, right numbers in the wrong order.
- Print `heap` after the expiry loop for each query of the second call: at q = 19 everything has ended, the heap is empty, and the answer is −1.

**Running bookings (My Calendar III).** The number of events open at time t is (starts ≤ t) − (ends ≤ t). Store only the changes, +1 at each start and −1 at each end; sweeping the keys in sorted order and summing rebuilds the count everywhere, and its peak is the answer. It is a difference array on a sparse, unbounded timeline. Each `book` below sorts the keys, O(n log n); keeping them in a sorted list with `bisect.insort` makes it O(n), and a lazy segment tree over the coordinate range C makes it O(log C).

In [ ]:
class MyCalendarThree:
    def __init__(self):
        self.delta = defaultdict(int)            # STATE: time -> change in the number of open events

    def book(self, start, end):                  # half-open [start, end)
        self.delta[start] += 1                   # STEP: one more event from start ...
        self.delta[end] -= 1                     # ... until end
        open_now = best = 0
        for t in sorted(self.delta):             # sweep the boundaries left to right
            open_now += self.delta[t]
            best = max(best, open_now)           # RECORD
        return best                              # RETURN


cal = MyCalendarThree()
print([cal.book(s, e) for s, e in [(10, 20), (50, 60), (10, 40), (5, 15), (5, 10), (25, 55)]])
# [1, 1, 2, 3, 3, 3]

**Try it**
- On a fresh calendar, book `(1, 5)` and then `(5, 9)`: both return 1. At time 5 the −1 and the +1 land on the same key and cancel, which is the half-open rule.
- Print `sorted(cal.delta.items())` after the six bookings and add the values up by hand: the running sum goes 2, 3, 2, 1, 2, 1, 2, 1, 0, so the peak is 3.
- My Calendar II (731) rejects a booking that would make a triple booking: add its two deltas, sweep, and if the peak reaches 3, take the two deltas back and return `False`.

**Area of a union (Rectangle Area II).** Area is a sum of thin strips. Sweep a vertical line through the rectangles' x-edges; between two consecutive edges the set of rectangles cut by the line does not change, so that strip adds (its width) × (the length of the union of their y-intervals). That length is the merge template again, in one dimension.

```text
  y
  3        +-----+                  rectangles [0,0,2,2] [1,0,2,3] [1,0,3,1]
  2  +-----+     |                  (x1, y1, x2, y2); the union's outline:
  1  |     |     +-----+
  0  +-----+-----+-----+--> x       strip      [0,1]  [1,2]  [2,3]
     0     1     2     3            covered y:   2      3      1      area = 2 + 3 + 1 = 6
```

In [ ]:
def covered_length(spans):                       # total length of a union of [lo, hi] spans
    total, reach = 0, -math.inf
    for lo, hi in sorted(spans):
        lo = max(lo, reach)                      # skip the part already counted
        if hi > lo:
            total += hi - lo
            reach = hi
    return total


def rectangle_area(rects):
    rects = [r for r in rects if r[0] < r[2] and r[1] < r[3]]    # a flat rectangle has no area
    if not rects:
        return 0
    events = sorted([(x1, 1, y1, y2) for x1, y1, x2, y2 in rects] +    # INIT: switch on at x1 ...
                    [(x2, -1, y1, y2) for x1, y1, x2, y2 in rects])    # ... and off at x2
    active, area, prev_x = [], 0, events[0][0]   # STATE: y-spans of the rectangles the line cuts
    for x, kind, y1, y2 in events:
        area += (x - prev_x) * covered_length(active)   # RECORD: the strip [prev_x, x] uses the OLD set
        prev_x = x
        if kind == 1:                            # STEP: then the event changes the set
            active.append((y1, y2))
        else:
            active.remove((y1, y2))
    return area % (10**9 + 7)                    # RETURN


print(rectangle_area([[0, 0, 2, 2], [1, 0, 2, 3], [1, 0, 3, 1]]))   # 6
print(rectangle_area([[0, 0, 1000000000, 1000000000]]))             # 49

**Try it**
- Apply the event *before* measuring the strip (move the `area += ...` and `prev_x = x` lines below the `if`/`else`): the first call gives 5 instead of 6. The last strip `[2, 3]` is now measured after `[1, 0, 3, 1]` has switched off, so it counts 0 instead of 1.
- Delete the filter line and run `rectangle_area([[0, 0, 1, 1], [1, 0, 1, 1]])`: `ValueError: list.remove(x): x not in list`. The flat rectangle's "off" event `(1, -1, ...)` sorts before its own "on" event.
- Why 49? The area is (10⁹)² = 10¹⁸, and 10¹⁸ mod (10⁹ + 7) = 49. Python integers don't overflow, so one modulo at the end is enough.

### Say it in the interview

> "Comparing every pair is O(n²). If I sort by start, an interval can only overlap the block I'm currently building, because everything after it starts even later. So I sort once and sweep once, keeping `merged[-1]` as the open block: O(n log n) time, O(n) for the output. For 'how many rooms', I keep a min-heap of the end times of running meetings: free the ended ones, add the newcomer, and the largest heap size is the answer. Fewer rooms is impossible, because at that moment that many meetings run at once; and it is enough, because a newcomer always finds a room whose meeting has ended."

Say which ends are closed *before* you write the comparison ("touching intervals merge, so `<=`"), and point at the `max` when you stretch: "max, because the next interval may be nested inside the block". Likely follow-ups and your answers:

- *Do touching intervals overlap?* → ask. Closed ends mean `<=`, half-open ends mean `<`.
- *The list is already sorted, insert one interval?* → three phases, O(n), no sort.
- *Which room does each meeting get?* → a heap of `(end, room)` plus a heap of free room ids (Meeting Rooms III).
- *Bookings arrive one by one?* → sorted starts plus `bisect` (My Calendar I), or the delta map for counts (My Calendar III).
- *Intersect two lists of intervals?* → two pointers; advance the one that ends first.
- *Fewest arrows, or keep the most intervals?* → sort by end.

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| Employee Free Time | `intervals/employee_free_time.py` | free time = the gaps between merged busy blocks; a heap merges the k sorted schedules in O(N log k) |
| Insert Interval | `intervals/insert_interval.py` | already sorted: copy what ends before, swallow what overlaps, copy the rest; O(n), no sort |
| Meeting Rooms II | `intervals/meeting_rooms_ii.py` · `practice/simple/49_meeting_rooms_ii.py` | sort by start; free ended rooms, push the end, count (or +1/−1 events, ends first); the peak is the answer |
| Merge Intervals | `intervals/merge_intervals.py` · `practice/simple/48_merge_intervals.py` | sort by start; only the last block can overlap; stretch its end with max |
| Minimum Interval to Include Each Query | `intervals/minimum_interval_to_include_each_query.py` | sort the queries offline; push started intervals by size; lazily pop the ended ones |
| My Calendar III | `intervals/my_calendar_iii.py` | difference map, +1 at start and −1 at end; the peak of the running sum is the answer |
| Non-overlapping Intervals | `intervals/non_overlapping_intervals.py` | keep the interval that ends first (exchange argument); count those starting before the kept end |
| Rectangle Area II | `intervals/rectangle_area_ii.py` | sweep x; each strip adds width × union length of the active y-intervals |

### Self-check

1. To keep the most non-overlapping intervals, why sort by *end*, and not by start or by length?
<details><summary>Answer</summary>The interval that ends first leaves the most room for everything after it, and any best answer can swap its first interval for that one without a clash. Start order can grab a long interval that blocks many (<code>[1, 100]</code>); length order fails too: in <code>[[1, 5], [4, 6], [5, 10]]</code> the shortest, <code>[4, 6]</code>, clashes with both others, while keeping <code>[1, 5]</code> and <code>[5, 10]</code> is better.</details>

2. In the rooms loop, why must `rooms = max(...)` come after both the freeing loop and the push?
<details><summary>Answer</summary>It reads how many meetings run at this instant. Before the freeing loop, meetings that already ended are still counted (<code>[1, 5]</code>, <code>[5, 9]</code> would need 2 rooms). Before the push, the newcomer is missing (<code>[[1, 5], [2, 6], [3, 7]]</code> would say 2 instead of 3).</details>

3. Write the overlap test for two closed intervals `a` and `b` without any `if`.
<details><summary>Answer</summary><code>a[0] &lt;= b[1] and b[0] &lt;= a[1]</code>. The common part is <code>[max(a[0], b[0]), min(a[1], b[1])]</code>, which is non-empty exactly when the test is true.</details>

4. Why does `merge` compare each interval with `merged[-1]` and not with the interval just before it?
<details><summary>Answer</summary>The block can reach further than the last interval: after <code>[1, 10]</code> and the nested <code>[2, 3]</code>, the previous end is 3 but the block ends at 10. Comparing with the previous interval would split <code>[4, 5]</code> off a block that already covers it.</details>

<a id="s15"></a>

## Greedy

> Make the move that looks best right now, never take it back, and carry only a tiny summary of the past: a running sum, the furthest reach, the fuel in the tank. Greedy is fast because it never branches; it is correct only when you can say why the greedy move never closes a door that the best answer needed.

**Reach for it when** one number can summarise everything that matters about the prefix you have seen (best sum ending here, furthest index reachable, fuel so far), the question asks for the minimum number of jumps / taps / patches / removals, sorting by one key (end time, deadline) makes "take the one that finishes first" obviously safe, or a small swap argument shows the locally best move is never worse. Typical words: "minimum number of …", "can you reach …", "maximum … subarray", "as many as possible".

**In this repo:** `greedy/` (14 problems) · greedy on intervals (Non-overlapping Intervals) is in the Intervals section; greedy with a heap (IPO, refuelling stops, Course Schedule III) is in the Heaps section · basics: `practice/simple/basics/graphs/03_union_find.py` (the cycle view of Couples Holding Hands), `practice/simple/basics/monotonic_stacks/05_remove_k_digits.py` (a greedy kept honest by a stack)

### The picture

```text
Kadane: extend, or restart when the past only hurts
nums:      -2    1   -3    4   -1    2    1   -5    4
cur:       -2    1   -2    4    3    5    6    1    5        cur = max(x, cur + x)
                 ^         ^              ^
              restart   restart        best = 6               (both restarts drop a carried -2)

Jump game: the reachable frontier only grows
nums  =  [2, 3, 1, 1, 4]
index:    0  1  2  3  4
i = 0:   [=======]          reach = 0 + 2 = 2: you can stand anywhere in 0..2
i = 1:   [=============]    reach = 1 + 3 = 4: the last index is covered -> True
```

Why it is fast: the brute force tries every choice: every subarray (O(n²)), every jump path (exponential), every subset of taps (2ⁿ). Greedy notices that one number summarises everything about the past that the future cares about, so each step is a single `max` or a single comparison: O(n), or O(n log n) with a sort in front.

### Can I trust the greedy choice?

With greedy the code is the easy part; *correctness* is the risk. Two plain-language arguments cover almost every interview greedy, plus one habit.

**Exchange argument** ("swap in the greedy choice; nothing gets worse"). Take any best answer. Find the first place where it differs from the greedy. Swap its choice for the greedy one and check that the answer stays valid and is no worse. Repeat, and the best answer turns into the greedy answer without ever getting worse. Example: to keep the most non-overlapping intervals, a best answer's first interval can be swapped for the interval that *ends* first; it ends no later, so it can't clash with anything kept after it.

**Greedy stays ahead** ("after every step, greedy is at least as far along"). Pick a measure of progress and show that after k steps the greedy is never behind any other strategy. Example: Jump Game II. After j jumps the greedy's reach is the furthest that *any* j jumps can get, because it took the best reach from everything reachable in j − 1 jumps. Ahead at every step means first to the end.

**The habit: test it against brute force.** Run the greedy against an exhaustive search on a few hundred tiny random inputs. A wrong greedy usually fails within seconds, and the failing input shows you *why* (see "When greedy fails" below).

```text
exchange:      best answer   [ X, ... ]        greedy picks G first
               swap X -> G   [ G, ... ]        still valid, no worse  ->  repeat

stays ahead:   step      1    2    3    4
               greedy    5    9   14   20      (reach, count, sum, ...)
               other     4    9   12   17      greedy >= other after every step
```

### From idea to code

**The idea in one sentence:** *walk left to right carrying the smallest summary of the past that decides the future; at each item make the locally best update (usually one `max`), and record the answer the moment it is known.*

| Decision | Greedy answer |
|---|---|
| **State**: what must I remember? | one or two numbers that summarise the prefix: `cur` (best sum ending here), `reach` (furthest index), `tank`, `end` of the current part |
| **Definition**: what exactly does each variable mean? | say it precisely, because the definition *is* the proof: `cur` = best sum of a subarray that ENDS at i; `reach` = furthest index reachable using indices 0..i |
| **Invariant**: what is true at the end of every step? | the summary is exact for the prefix 0..i, and nothing else about the prefix can matter later |
| **Step**: how does one item change the state? | one local choice, usually a `max`: `cur = max(x, cur + x)`, `reach = max(reach, i + nums[i])` |
| **Record**: when is the answer updated? | right after the step (`best = max(best, cur)`), or when a boundary is hit (`i == end`: cut a part, count a jump) |
| **Init**: starting values | the first item (`cur = best = nums[0]`) or a neutral value (`reach = 0`); never `best = 0` if every value can be negative |
| **Return**: what comes back, and for "not found"? | `best`, a count, or the failure value: `False` when `i > reach`, −1 when the total gas is less than the total cost |

The same idea, sentence by sentence:

| In words | In code |
|---|---|
| "best sum ending here: extend, or start fresh" | `cur = max(x, cur + x)` |
| "best so far" | `best = max(best, cur)` |
| "the furthest I can get" | `reach = max(reach, i + nums[i])` |
| "I'm standing where I can't be" | `if i > reach: return False` |
| "this jump's range is used up" | `if i == cur_end: jumps += 1; cur_end = farthest` |
| "the tank ran dry: start after here" | `if tank < 0: start, tank = i + 1, 0` |
| "this part must reach every letter's last copy" | `end = max(end, last[c])` |

Two templates: a running best (Kadane), and a frontier counted in levels (Jump Game II).

In [ ]:
def max_subarray(nums):
    cur = best = nums[0]                     # STATE + INIT: cur = best sum of a subarray ENDING here
    for x in nums[1:]:
        cur = max(x, cur + x)                # STEP: extend, or restart if the past only hurts
        best = max(best, cur)                # RECORD
    return best                              # RETURN


def min_jumps(nums):
    jumps = cur_end = farthest = 0           # STATE + INIT: cur_end = edge of what `jumps` jumps reach
    for i in range(len(nums) - 1):           # stop before the last index: arriving needs no jump
        farthest = max(farthest, i + nums[i])    # STEP: the best reach from this level
        if i == cur_end:                     # RECORD: this level is used up, one more jump
            jumps += 1
            cur_end = farthest
    return jumps                             # RETURN


print(max_subarray([-2, 1, -3, 4, -1, 2, 1, -5, 4]), max_subarray([-3, -1, -2]))   # 6 -1
print(min_jumps([2, 3, 1, 1, 4]), min_jumps([0]))                                   # 2 0

**Try it**
- Start from `cur = best = 0` and loop over all of `nums`: `max_subarray([-3, -1, -2])` returns 0, a sum that no non-empty subarray has.
- Loop `for i in range(len(nums))` (one step too far) and run `min_jumps([1, 1])`: 2 instead of 1. Standing on the last index triggered a jump to nowhere.
- Print `i, farthest, cur_end, jumps` inside the loop for `[2, 3, 1, 1, 4]`: `jumps` changes only when `i` reaches `cur_end`.
- Rewrite the step as `cur = cur + x if cur > 0 else x`: same answers. "Restart when the carried sum is not positive" is the same rule in other words.

### Watch it work

In [ ]:
def trace_kadane(nums):
    cur = best = nums[0]
    print(f"x={nums[0]:>2}  cur={cur:>2}  best={best:>2}  (start)")
    for x in nums[1:]:
        note = "restart: the carried sum was negative" if cur < 0 else "extend"
        cur = max(x, cur + x)
        best = max(best, cur)
        print(f"x={x:>2}  cur={cur:>2}  best={best:>2}  {note}")


def trace_jumps(nums):
    jumps = cur_end = farthest = 0
    for i in range(len(nums) - 1):
        farthest = max(farthest, i + nums[i])
        if i == cur_end:
            jumps += 1
            cur_end = farthest
            print(f"i={i}: level used up -> jump {jumps}, the next level ends at {cur_end}")
    print("jumps:", jumps)


trace_kadane([-2, 1, -3, 4, -1, 2, 1, -5, 4])
trace_jumps([2, 3, 1, 1, 4])

**Try it**
- Run `trace_kadane([5, -9, 6])`: the 5 is carried into −4, then dropped, and `cur` restarts at 6, which is also the best.
- Run `trace_kadane([-3, -1, -2])`: every step restarts, and `best` ends as the single largest number, −1.
- Run `trace_jumps([1, 1, 1, 1])`: every level is a single index, so 3 jumps. Then `trace_jumps([4, 1, 1, 1, 1])`: one jump covers everything, and the loop scans on without jumping again.

### Where it goes wrong

1. **Starting `best` (or restarting) at 0.** Kadane on `[-3]` returns 0. Start from `nums[0]` and restart to `x`, not to 0.
2. **Checking reachability after updating it.** In Jump Game, updating `reach` before testing `i > reach` lets you jump *from* an index you never reached: `[0, 1]` becomes `True`.
3. **One loop step too many.** Jump Game II stops at index n − 2; standing on the last index needs no jump (`[1, 1]` gives 2 instead of 1).
4. **Updating `hi` before computing `lo`** in Maximum Product. Compute both from the *old* pair in one tuple assignment; the line-by-line version turns `[-1, -2, -1]` into 4, a product no subarray has (the answer is 2).
5. **Overwriting instead of `max` in Candy's second pass.** The right-to-left pass must keep what the left pass needed: `[1, 2, 3, 1]` gives 6 instead of 7.
6. **Restarting at `i` instead of `i + 1`** in Gas Station: the station where the tank went negative belongs to the doomed stretch.
7. **Trusting a greedy without a proof or a test.** "Biggest coin first" fails for coins `[1, 3, 4]` and amount 6: 4 + 1 + 1 instead of 3 + 3.

### Edge cases to say out loud

One element · all negative (Kadane) · zeros (Maximum Product restarts there) · `[0]` (already at the end: reachable, 0 jumps) · a 0 you must cross (`[3, 2, 1, 0, 4]` is stuck) · total gas below total cost · equal neighbours (Candy: no rule between them) · ties in the sort order (Set Intersection: by end, then start descending).

In [ ]:
assert max_subarray([5]) == 5
assert max_subarray([-3, -1, -2]) == -1                  # all negative: never 0
assert max_subarray([0, -1, 0]) == 0
assert max_subarray([2, -1, 2]) == 3                     # a dip worth crossing
assert min_jumps([0]) == 0                               # already at the end
assert min_jumps([1, 1]) == 1
assert min_jumps([5, 1, 1, 1]) == 1                      # one jump covers everything
assert min_jumps([1, 2, 1, 1, 1]) == 3
print("edge cases pass")

**Try it**
- Predict `max_subarray([-1, 5, -1])` (5), then add the assert.
- What should `min_jumps([2, 1])` be? One jump: from index 0 you reach index 1 directly. Add it.
- `min_jumps` assumes the end is reachable. Call `min_jumps([1, 0, 1])`: it returns 2, a wrong answer rather than an error. If the input can be stuck, add Jump Game's `i > reach` check.

### Variations

| Variation | What changes from the template | Problems |
|---|---|---|
| **Extend or restart** | `cur = max(x, cur + x)` | 53 |
| **Signs flip the order** | carry the largest AND the smallest product ending here | 152 |
| **Reachable frontier** | one number `reach`; stuck when `i > reach` | 55 |
| **Frontier in levels** | `cur_end` and `farthest`; count a jump (a tap) when `i == cur_end` | 45, 1326 |
| **Running sum with restart** | tank < 0 after i: no start in that stretch works, so restart at i + 1 | 134 |
| **Intervals in disguise** | each letter spans [first, last]; cut where no span crosses | 763 |
| **Two passes** | satisfy the left neighbours, then the right ones; keep the max | 135 |
| **Covered prefix** | sums 1..reach are covered; patch reach + 1 to double it | 330 |
| **Work backwards** | undo the last move first: it is the one still fully visible | 936 |
| **Rightmost points** | sort by end; new points go at the right end, where they serve the most later intervals | 757 |
| **Bounds that meet** | the answer is the largest lower bound, and it is always achievable | 517, 1526 |
| **Swap into place** | fix positions left to right; one swap per link of a cycle | 765 |

**Signs flip the order (Maximum Product Subarray).** A negative number turns the largest product into the smallest and the smallest into the largest. So carry both: whichever one the next negative flips into the lead is already in hand. A zero resets both to 0, a fresh start.

In [ ]:
def max_product(nums):
    hi = lo = best = nums[0]                 # hi / lo = largest / smallest product ENDING here
    for x in nums[1:]:
        hi, lo = max(x, hi * x, lo * x), min(x, hi * x, lo * x)   # both from the OLD hi and lo
        best = max(best, hi)
    return best


print(max_product([2, 3, -2, 4]), max_product([-2, 0, -1]), max_product([-2, 3, -4]))   # 6 0 24

**Try it**
- Split the tuple assignment into two lines (`hi = ...`, then `lo = ...`): `max_product([-1, -2, -1])` gives 4, a product no subarray has (the answer is 2). `lo` was computed from the *new* `hi`.
- Print `hi, lo` after each step for `[-2, 3, -4]`: `(3, -6)`, then `(24, -12)`. The −6 waited in `lo` until −4 flipped it into the lead.
- Keep only `hi` (plain Kadane with products, `hi = max(x, hi * x)`): `[-2, 3, -4]` gives 3 instead of 24.

**Reachable frontier, and covering with the fewest intervals (Jump Game, Taps).** In Jump Game the indices you can reach always form a prefix `[0, reach]`, so one number is the whole state. The taps problem looks different but is Jump Game II in disguise: bucket every tap by its left end, `reach[l]` = the furthest right end of a tap that starts at `l`, then count the "jumps" needed to cover `[0, n]`.

```text
garden [0, 5], ranges [3, 4, 1, 1, 0, 0]:  tap i waters [i - r, i + r], clipped at 0
tap 0: [0, 3]   tap 1: [0, 5]   tap 2: [1, 3]   tap 3: [2, 4]   taps 4, 5: single points
reach[0] = 5, so one tap (tap 1) waters everything
```

In [ ]:
def can_jump(nums):
    reach = 0                                # furthest index reachable so far
    for i, step in enumerate(nums):
        if i > reach:                        # I can't even stand here
            return False
        reach = max(reach, i + step)
    return True


def min_taps(n, ranges):
    reach = [0] * (n + 1)                    # reach[l] = furthest right end of a tap starting at l
    for i, r in enumerate(ranges):
        left = max(0, i - r)
        reach[left] = max(reach[left], i + r)
    taps = cur_end = farthest = 0            # exactly min_jumps, with reach[i] for i + nums[i]
    for i in range(n):
        farthest = max(farthest, reach[i])
        if i == cur_end:                     # the watered prefix ends here: open one more tap
            if farthest <= i:                # nothing waters past i: a dry gap
                return -1
            taps += 1
            cur_end = farthest
    return taps


print(can_jump([2, 3, 1, 1, 4]), can_jump([3, 2, 1, 0, 4]))        # True False
print(min_taps(5, [3, 4, 1, 1, 0, 0]), min_taps(3, [0, 0, 0, 0]))  # 1 -1
print(min_taps(7, [1, 2, 1, 0, 2, 1, 0, 1]))                       # 3

**Try it**
- In `can_jump`, move the `reach = ...` line above the `if`: `can_jump([0, 1])` becomes `True`. You jumped from index 1 without ever standing on it.
- Drop the clamp (`left = i - r`) and rerun the first `min_taps` call: −1 instead of 1. `reach[-3]` silently wrote into index 3 (negative indexing), so no tap seems to start at 0.
- `min_taps(0, [5])` is 0: a garden that is a single point needs no tap, and the loop never runs.
- Print `i, farthest, cur_end, taps` in `min_taps` for the last call and compare with the `min_jumps` trace: it is the same machine.

**Running sum with restart (Gas Station) and intervals in disguise (Partition Labels).** Two one-pass greedies where a single running number decides where to cut.

*Gas Station:* if you start at s and the tank first goes negative after station i, then *no* start between s and i works either: you reached each of them with a non-negative tank, so starting there with an empty one is no better. Jump the candidate start straight to i + 1. If the total gas covers the total cost, the last candidate is the answer.

*Partition Labels:* every letter spans from its first to its last copy, and a part must swallow the whole span of every letter it touches. Sweep left to right pushing the part's end out to `last[c]`; when `i` catches up with the end, no letter inside reaches further: cut.

In [ ]:
def can_complete_circuit(gas, cost):
    total = tank = start = 0
    for i in range(len(gas)):
        diff = gas[i] - cost[i]
        total += diff                        # balance of the whole loop
        tank += diff                         # balance since the candidate start
        if tank < 0:                         # dry after i: no start in start..i can work
            start, tank = i + 1, 0
    return start if total >= 0 else -1


def partition_labels(s):
    last = {c: i for i, c in enumerate(s)}   # each letter's span ends at its last copy
    sizes, start, end = [], 0, 0
    for i, c in enumerate(s):
        end = max(end, last[c])              # this part must reach every letter it has touched
        if i == end:                         # nothing inside reaches further: cut here
            sizes.append(end - start + 1)
            start = i + 1
    return sizes


print(can_complete_circuit([1, 2, 3, 4, 5], [3, 4, 5, 1, 2]), can_complete_circuit([2, 3, 4], [3, 4, 3]))   # 3 -1
print(partition_labels("ababcbacadefegdehijhklij"))                                                         # [9, 7, 8]

**Try it**
- Restart at `i` instead of `i + 1` (`start, tank = i, 0`): the first call returns 2, and starting at station 2 dies at once (3 gas, 5 cost).
- Print `start, tank` at the end of each step of the first call: the start jumps to 1, 2, 3 while the tank keeps dying, then the tank climbs to 3 and 6.
- In `partition_labels`, test `i == end` *before* updating `end`: `partition_labels("abab")` gives `[1, 3]` instead of `[4]`. The first `a` was cut off before its span was known.
- Predict, then run: `partition_labels("abc")` is `[1, 1, 1]` and `partition_labels("abca")` is `[4]`.

**Two passes (Candy).** Each child has two rules: more candy than a lower-rated left neighbour, and more than a lower-rated right neighbour. One left-to-right pass satisfies every left rule with the fewest candies; one right-to-left pass does the same for the right rules. A child must satisfy both, so it takes the larger of its two counts.

```text
ratings:      1   2   3   1   0
left pass:    1   2   3   1   1      climb from the left
right pass:   1   1   3   2   1      climb from the right
max of both:  1   2   3   2   1      = 9 candies
```

In [ ]:
def candy(ratings):
    n = len(ratings)
    c = [1] * n
    for i in range(1, n):                    # left -> right: beat a lower left neighbour
        if ratings[i] > ratings[i - 1]:
            c[i] = c[i - 1] + 1
    for i in range(n - 2, -1, -1):           # right -> left: beat a lower right neighbour too
        if ratings[i] > ratings[i + 1]:
            c[i] = max(c[i], c[i + 1] + 1)   # max: keep what the left rule needed
    return sum(c)


print(candy([1, 0, 2]), candy([1, 2, 2]), candy([1, 2, 3, 1, 0]))   # 5 4 9

**Try it**
- Replace `max(c[i], c[i + 1] + 1)` with `c[i + 1] + 1`: `candy([1, 2, 3, 1])` gives 6 instead of 7. The peak lost the 3 it needed for its left side.
- Print `c` after each pass for `[1, 2, 3, 1, 0]` and compare with the picture.
- `candy([2, 2, 2])` is 3: equal neighbours have no rule between them, so everyone gets 1.

**Covered prefix (Patching Array).** Keep `reach` with the promise "every value in 1..reach is a subset sum". A number x ≤ reach + 1 glues onto that prefix and extends it to 1..reach + x. A bigger x would leave reach + 1 as a hole that no later (even bigger) number can fill, so patch it, and the best patch is reach + 1 itself: it doubles the coverage, the most any single number can add.

In [ ]:
def min_patches(nums, n):
    reach, patches, i = 0, 0, 0              # every value in 1..reach is a subset sum
    while reach < n:
        if i < len(nums) and nums[i] <= reach + 1:   # glues on: now 1..reach + nums[i]
            reach += nums[i]
            i += 1
        else:                                # reach + 1 is a hole: patch it, coverage doubles
            reach += reach + 1
            patches += 1
    return patches


print(min_patches([1, 3], 6), min_patches([1, 5, 10], 20), min_patches([1, 2, 2], 5))   # 1 2 0

**Try it**
- Change `nums[i] <= reach + 1` to `nums[i] <= reach`: `min_patches([1, 3], 6)` gives 2 instead of 1. The first patch is a 1, although the array already had one.
- Print `reach` after each patch of `min_patches([], 20)`: 1, 3, 7, 15, 31. With no numbers at all, patching 1, 2, 4, 8, 16 doubles the coverage each time: 5 patches.
- Patch a smaller value instead, say always 1 (`reach += 1`): `min_patches([], 20)` needs 20 patches instead of 5.

**Work backwards (Stamping the Sequence).** Going forwards you can't know which stamps will be covered later. Going backwards it is easy: the *last* stamp is still fully visible, so find a window equal to the stamp and peel it off, turning its letters into `?` wildcards (whatever lay underneath was overwritten anyway). Peeling only turns letters into wildcards, so it can only make other windows easier to match; that is why "peel anything that matches" is safe. Reverse the peel order to get the stamp order.

```text
target   a b a b c        stamp "abc"
peel 2:  a b ? ? ?        window 2 is exactly "abc"
peel 0:  ? ? ? ? ?        window 0 is "ab?": '?' matches anything
stamp order = the peel order reversed = [0, 2]
```

In [ ]:
def moves_to_stamp(stamp, target):
    m, n = len(stamp), len(target)
    t, erased, peeled = list(target), 0, []
    progress = True
    while erased < n and progress:
        progress = False
        for i in range(n - m + 1):
            window = t[i:i + m]
            fits = all(ch == "?" or ch == sc for ch, sc in zip(window, stamp))
            if fits and window != ["?"] * m:     # matches, and still has a real letter to peel
                for j in range(i, i + m):
                    if t[j] != "?":
                        t[j] = "?"
                        erased += 1
                peeled.append(i)
                progress = True
    return peeled[::-1] if erased == n else []   # the last peel was the first stamp


print(moves_to_stamp("abc", "ababc"), moves_to_stamp("abca", "aabcaca"), moves_to_stamp("ab", "ba"))
# [0, 2] [0, 3, 1] []

**Try it**
- Check the second answer forwards by hand: from `???????`, stamp `abca` at 0, 3, 1: `abca???`, `abcabca`, `aabcaca`. That is the target.
- Return `peeled` without reversing it: `[1, 3, 0]`, and applying it forwards gives `abcabca`, not the target. The order matters.
- Add `print(i, "".join(t))` after each peel for the second call and watch the target dissolve into `?`s.
- Why the `window != ["?"] * m` test? Without it, an all-`?` window "matches" on every pass and progress never stops when a target is impossible, as in `moves_to_stamp("ab", "abx")`. Reason it out rather than running it: that loop would never end.

**Rightmost points (Set Intersection Size At Least Two).** Process intervals by end. When an interval still lacks points, add them as far right as possible, at e − 1 and e: every later interval ends at or after e, so a point at the right end is shared with as many of them as possible. Only the two largest chosen points can matter later, so they are the whole state. On equal ends, take the *narrower* interval first (start descending): its points then serve the wider one too.

In [ ]:
def intersection_size_two(intervals):
    a, b, count = -math.inf, -math.inf, 0    # the two largest chosen points, a < b
    for s, e in sorted(intervals, key=lambda iv: (iv[1], -iv[0])):   # by end; equal ends: narrow first
        if s > b:                            # neither point is inside: add e - 1 and e
            a, b = e - 1, e
            count += 2
        elif s > a:                          # only b is inside: add e
            a, b = b, e
            count += 1
    return count


print(intersection_size_two([[1, 3], [3, 7], [8, 9]]), intersection_size_two([[1, 3], [1, 4], [2, 5], [3, 5]]))   # 5 3

**Try it**
- Break ties by start *ascending* (`key=lambda iv: (iv[1], iv[0])`) and run `intersection_size_two([[0, 1], [1, 5], [3, 5], [5, 6]])`: 4 instead of 5. The wide `[1, 5]` picks 5, then `[3, 5]` "adds" 5 again, a duplicate.
- Print `a, b` after each interval of the first call: `(2, 3)`, `(3, 7)`, `(8, 9)`. The chosen set is {2, 3, 7, 8, 9}.
- Change `s > b` to `s >= b`: the first call gives 6 instead of 5. The intervals are closed, so a point sitting exactly at `s` is inside.

**Bounds that meet (Super Washing Machines, Minimum Increments).** Sometimes the greedy *is* a formula: find the lower bounds every schedule must respect, and show the largest one can always be reached. Washing machines: the dresses that must cross each boundary (the prefix balance) cross at most one per move, and an overloaded machine can hand out only one dress per move. Minimum increments: every +1 stroke has exactly one left edge, and a step up of d from the left neighbour needs d new left edges there.

In [ ]:
def find_min_moves(machines):
    total, n = sum(machines), len(machines)
    if total % n:
        return -1                            # can't split evenly
    target = total // n
    balance = best = 0
    for load in machines:
        excess = load - target               # this machine must hand out `excess`, one per move
        balance += excess                    # dresses that must cross the boundary on its right
        best = max(best, abs(balance), excess)
    return best


def min_number_operations(target):           # strokes start only where the skyline steps up
    return target[0] + sum(max(0, b - a) for a, b in zip(target, target[1:]))


print(find_min_moves([1, 0, 5]), find_min_moves([0, 3, 0]), find_min_moves([0, 2, 0]))   # 3 2 -1
print(min_number_operations([3, 1, 5, 4, 2, 3, 4, 2]))                                   # 9

**Try it**
- Drop `excess` from the `max`: `find_min_moves([0, 3, 0])` gives 1 instead of 2. The middle machine has two dresses to give and can only pass one per move.
- Use `abs(excess)` instead: `find_min_moves([3, 0, 3])` gives 2 instead of 1. A machine that is *short* can receive from both sides in the same move.
- List the steps up for `[3, 1, 5, 4, 2, 3, 4, 2]`: 3 at the start, then 4, 1, 1. That is 9 strokes.

**Swap into place (Couples Holding Hands).** Fix the couches from left to right: if the person on seat `i + 1` is not the partner of the person on seat `i`, swap the partner in. Each swap seats one couple for good. Why is that the minimum? Draw each couple as a node and each couch as an edge between the two couples sitting on it: the seating splits into cycles, a cycle of k couples needs exactly k − 1 swaps, and the left-to-right fixing never wastes one. So the answer is (couples − cycles).

In [ ]:
def min_swaps_couples(row):
    row = row[:]                             # don't edit the caller's list
    seat = {p: i for i, p in enumerate(row)} # where each person sits now
    swaps = 0
    for i in range(0, len(row), 2):          # couch (i, i + 1), left to right
        partner = row[i] ^ 1                 # 0 <-> 1, 2 <-> 3, ...
        if row[i + 1] != partner:
            j = seat[partner]                # bring the partner next to row[i]
            row[i + 1], row[j] = row[j], row[i + 1]
            seat[row[j]], seat[row[i + 1]] = j, i + 1
            swaps += 1
    return swaps


print(min_swaps_couples([0, 2, 1, 3]), min_swaps_couples([3, 2, 0, 1]), min_swaps_couples([0, 2, 4, 1, 3, 5]))   # 1 0 2

**Try it**
- Print `row` after every swap for `[0, 2, 4, 1, 3, 5]`: `[0, 1, 4, 2, 3, 5]`, then `[0, 1, 4, 5, 3, 2]`. Each swap completes one couch for good.
- Delete the line that updates `seat` and run `min_swaps_couples([0, 2, 3, 4, 1, 5])`: 3 instead of 2. A stale seat number sent a swap to the wrong place.
- Check the partner trick: `[x ^ 1 for x in range(6)]` is `[1, 0, 3, 2, 5, 4]`.

### When greedy fails, and what to do

Greedy fails when today's best move blocks a better future. Coin change is the classic: with coins `[1, 3, 4]`, "biggest coin first" pays 6 as 4 + 1 + 1 (three coins) instead of 3 + 3 (two). Don't stare harder at the greedy: *test* it against a brute force on small inputs.

In [ ]:
from itertools import combinations_with_replacement


def coins_greedy(coins, amount):
    count = 0
    for c in sorted(coins, reverse=True):    # biggest coin first, as many as fit
        count += amount // c
        amount %= c
    return count if amount == 0 else -1


def coins_brute(coins, amount):              # fewest coins: try 0, 1, 2, ... coins
    for k in range(amount + 1):
        if any(sum(combo) == amount for combo in combinations_with_replacement(coins, k)):
            return k
    return -1


def first_counterexample(fast, slow, coins, amounts):
    for amount in amounts:
        if fast(coins, amount) != slow(coins, amount):
            return amount, fast(coins, amount), slow(coins, amount)
    return None


print(first_counterexample(coins_greedy, coins_brute, [1, 3, 4], range(1, 15)))       # (6, 3, 2)
print(first_counterexample(coins_greedy, coins_brute, [1, 5, 10, 25], range(1, 15)))  # None

**Try it**
- Predict, then run, the coins `[1, 7, 10]`: the first counterexample is `(14, 5, 2)`, greedy 10 + 1 + 1 + 1 + 1 against 7 + 7.
- Print `combo` when `coins_brute` succeeds for `[1, 3, 4]` and amount 6: it is `(3, 3)`, found at k = 2.
- Point `first_counterexample`'s idea at a greedy of your own: for Non-overlapping Intervals, write "try every subset" and compare it with the sort-by-start version. It fails on `[[1, 100], [2, 3], [4, 5]]`.

What to do when the test breaks the greedy:

1. **Try another greedy key.** Many greedies fail with one sort key and work with another: intervals by start fail, by end work; courses by duration fail, by deadline work.
2. **Greedy with regret.** Take the greedy move but keep your choices in a heap, so a later conflict can undo the worst one (Course Schedule III, refuelling stops: Heaps section).
3. **Search with pruning.** If choices really interact, explore them: backtracking with good pruning (Backtracking section).
4. **Remember sub-answers.** If the same sub-question keeps coming back ("fewest coins for amount a"), that is dynamic programming, and coin change is its textbook example.

### Say it in the interview

> "The brute force tries every subarray: O(n²). But the best sum of a subarray ending at i only depends on the best sum ending at i − 1: extend it if it helps, otherwise start fresh at i. So I carry one number and update it with a max: O(n) time, O(1) space. It's safe because a negative running sum can only lower whatever it is added to, so dropping it never loses the optimum."

For any greedy, say the choice *and* the reason in one breath: "I always keep the interval that ends first, because swapping any best answer's first interval for it never causes a clash." If you can't say the reason, try two or three tiny inputs (ties, zeros, negatives) to break it before you code.

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| Candy | `greedy/candy.py` | left pass beats the left neighbour, right pass the right one; each child takes the max of both |
| Couples Holding Hands | `greedy/couples_holding_hands.py` | couples are nodes, couches are edges: swaps = couples − cycles; fixing couches left to right achieves it |
| Gas Station | `greedy/gas_station.py` | if the tank dies after i, no start up to i works: restart at i + 1; possible iff total ≥ 0 |
| Jump Game | `greedy/jump_game.py` | reachable indices form a prefix: track reach, fail as soon as i > reach |
| Jump Game II | `greedy/jump_game_ii.py` | BFS whose levels are intervals: jump when i reaches cur_end, then cur_end = farthest |
| Maximum Product Subarray | `greedy/maximum_product_subarray.py` | a negative swaps the max and the min: carry both products ending here |
| Maximum Subarray | `greedy/maximum_subarray.py` | Kadane: cur = max(x, cur + x); a negative running sum is dead weight |
| Minimum Number of Increments on Subarrays to Form a Target Array | `greedy/min_number_operations.py` | every stroke has one left edge: target[0] + the sum of the positive steps up |
| Minimum Number of Taps to Open to Water a Garden | `greedy/minimum_number_of_taps_to_open_to_water_a_garden.py` | bucket taps by left end, then Jump Game II over reach; −1 when the frontier stalls |
| Partition Labels | `greedy/partition_labels.py` | each letter spans [first, last]; push end to last[c], cut when i == end |
| Patching Array | `greedy/patching_array.py` | sums 1..reach are covered; x ≤ reach + 1 extends it, otherwise patch reach + 1 (doubling) |
| Set Intersection Size At Least Two | `greedy/set_intersection_size_at_least_two.py` | sort by end (ties: start descending); keep the two largest points; add missing ones at e − 1, e |
| Stamping The Sequence | `greedy/stamping_the_sequence.py` | work backwards: peel any window matching the stamp ('?' is a wildcard), then reverse the order |
| Super Washing Machines | `greedy/super_washing_machines.py` | answer = max over machines of abs(prefix balance) and own excess; both are lower bounds, both reachable |

### Self-check

1. In Kadane, why is it safe to throw away a negative running sum?
<details><summary>Answer</summary>Any subarray that ends later and keeps the negative carried part would be strictly larger without it. So the best subarray ending at the next index is either that number alone or an extension of a non-negative carry: dropping a negative prefix never loses the optimum.</details>

2. Jump Game II loops `for i in range(len(nums) - 1)`. Why not up to the last index?
<details><summary>Answer</summary>A jump is counted when the scan reaches the end of the current level. Arriving at the last index needs no further jump, but if a level ends exactly there, scanning that index would count one more jump (<code>[1, 1]</code> would give 2).</details>

3. Why does Candy's second pass use `max` instead of overwriting?
<details><summary>Answer</summary>The left pass already set the smallest count that satisfies the left neighbour. Overwriting with the right-pass value can lower it and break that rule: in <code>[1, 2, 3, 1]</code> the peak needs 3 because of its left side but only 2 because of its right side.</details>

4. In an interview, how do you check a greedy you are not sure about?
<details><summary>Answer</summary>Attack it with tiny inputs first: ties, zeros, negatives, one huge item, and the input where a "bigger now" choice blocks the future. If it survives, state the exchange or stays-ahead argument in one sentence. When practising, run it against a brute force on a few hundred random small inputs.</details>